# Image branch and fusion: EfficientNet-B0, YOLOv8s weapons, CLIP probe, late fusion

Part of **Multimodal Content Moderation** (open re-implementation on public data).

**Before you click Run All**
1. Settings (right panel): *Accelerator* = **GPU T4 x2** (or P100), *Internet* = **On**, *Persistence* off.
2. *Add Input*: `williamberrios/hateful-memes` (Hateful Memes phase-2 release with labelled `test_seen`; accept Meta's licence at the official source first) and `reihanenamdari/youtube-toxicity-data` (copied into the zip for the local out-of-domain test). Open Images weapons download automatically.
3. Click **Run All**. When it finishes, open the *Output* tab and download **`results_image.zip`**.
4. On your laptop: `python -m scripts.merge_results results_image.zip` inside the repo.

Every number is written to `results/` by the code; nothing is typed by hand.


In [ ]:
# Cell 1: write the project source into /kaggle/working (generated by scripts/build_notebooks.py)
import os, sys, pathlib
WORK = pathlib.Path("/kaggle/working/multimodal-moderation") if pathlib.Path("/kaggle").exists() else pathlib.Path("multimodal-moderation").resolve()
FILES = {'src/__init__.py': '', 'src/config.py': '"""Shared paths, labels, seeds and small I/O helpers."""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport random\nfrom pathlib import Path\nfrom typing import Any\n\n# MM_ROOT overrides the repo root (the Kaggle notebooks point it at their working copy).\nROOT = Path(os.environ.get("MM_ROOT", Path(__file__).resolve().parents[1]))\nDATA_RAW = ROOT / "data" / "raw"\nDATA_PROCESSED = ROOT / "data" / "processed"\nRESULTS = ROOT / "results"\nMODELS = ROOT / "models"\n\nSEED = 42\nLABELS = ["Safe", "Neutral", "Harmful"]\nLABEL2ID = {name: i for i, name in enumerate(LABELS)}\n\n# Minimum Harmful recall the triage threshold must reach on validation data.\nTARGET_RECALL = 0.90\n\n\ndef set_seed(seed: int = SEED) -> None:\n    """Seed python, numpy and (if installed) torch."""\n    random.seed(seed)\n    os.environ["PYTHONHASHSEED"] = str(seed)\n    try:\n        import numpy as np\n\n        np.random.seed(seed)\n    except ImportError:  # pragma: no cover\n        pass\n    try:\n        import torch\n\n        torch.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n    except ImportError:\n        pass\n\n\ndef _to_builtin(obj: Any) -> Any:\n    """Convert numpy scalars/arrays so json.dump works."""\n    try:\n        import numpy as np\n\n        if isinstance(obj, np.generic):\n            return obj.item()\n        if isinstance(obj, np.ndarray):\n            return obj.tolist()\n    except ImportError:  # pragma: no cover\n        pass\n    if isinstance(obj, dict):\n        return {str(k): _to_builtin(v) for k, v in obj.items()}\n    if isinstance(obj, (list, tuple)):\n        return [_to_builtin(v) for v in obj]\n    return obj\n\n\ndef write_json(path: Path | str, payload: dict, merge: bool = True) -> Path:\n    """Write a results file. With merge=True, top-level keys are merged into an existing file."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    data: dict = {}\n    if merge and path.exists():\n        data = json.loads(path.read_text(encoding="utf-8"))\n    data.update(_to_builtin(payload))\n    path.write_text(json.dumps(data, indent=2, sort_keys=False) + "\\n", encoding="utf-8")\n    return path\n\n\ndef read_json(path: Path | str) -> dict:\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n', 'src/eval/__init__.py': '', 'src/eval/latency.py': '"""Latency benchmark against a running API (run on the laptop CPU).\n\nUsage:\n    uvicorn src.serve.app:app --port 8000 &          # in another terminal\n    python -m src.eval.latency --mode text --n 500\n    python -m src.eval.latency --mode fusion --n 300 --memes-root data/raw/hateful_memes\n\nSends sequential requests (one item each, after warm-up) and records wall-clock time per request\nas seen by the client, plus the server-reported model time. Also measures throughput with\nconcurrent clients. Writes results/latency.json and results/latency_samples.json.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport platform\nimport statistics\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\n\nimport numpy as np\n\nfrom src.config import RESULTS, SEED, write_json\n\n\ndef cpu_info() -> dict:\n    info = {\n        "machine": platform.machine(),\n        "system": platform.system(),\n        "python": platform.python_version(),\n        "logical_cores": os.cpu_count(),\n    }\n    try:\n        import cpuinfo  # optional: py-cpuinfo\n\n        info["cpu"] = cpuinfo.get_cpu_info().get("brand_raw")\n    except ImportError:\n        info["cpu"] = platform.processor() or None\n    try:\n        import torch\n\n        info["torch_threads"] = torch.get_num_threads()\n    except ImportError:\n        pass\n    return info\n\n\ndef pct(xs, q) -> float:\n    return round(float(np.percentile(xs, q)), 2)\n\n\ndef main() -> None:\n    import httpx\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--url", default="http://127.0.0.1:8000")\n    ap.add_argument("--mode", choices=["text", "fusion"], default="text")\n    ap.add_argument("--n", type=int, default=500)\n    ap.add_argument("--warmup", type=int, default=20)\n    ap.add_argument("--concurrency", type=int, default=8)\n    ap.add_argument("--memes-root")\n    args = ap.parse_args()\n\n    if args.mode == "text":\n        from src.text.data import load_jigsaw\n\n        texts = load_jigsaw()["test"]["text"].sample(args.n + args.warmup, random_state=SEED).tolist()\n        payloads = [({"text": t}, None) for t in texts]\n    else:\n        from src.image.memes import load_memes\n\n        s, _ = load_memes(args.memes_root)\n        from pathlib import Path\n\n        pool = s["test"][s["test"]["img_path"].map(lambda p: Path(p).exists())]  # local copies may be a subset\n        df = pool.sample(min(args.n + args.warmup, len(pool)), random_state=SEED)\n        payloads = [({"text": r.text}, open(r.img_path, "rb").read()) for r in df.itertuples()]\n\n    client = httpx.Client(base_url=args.url, timeout=60)\n    health = client.get("/health").json()\n\n    def call(pl):\n        """Send one request; returns (client wall-clock ms, server-reported latency_ms)."""\n        data, img = pl\n        files = {"image": ("x.png", img, "image/png")} if img is not None else None\n        t0 = time.perf_counter()\n        r = client.post("/moderate", data=data, files=files)\n        r.raise_for_status()\n        return (time.perf_counter() - t0) * 1000, r.json()["latency_ms"]\n\n    for pl in payloads[: args.warmup]:\n        call(pl)\n    rows = [call(pl) for pl in payloads[args.warmup :]]\n    client_ms = [r[0] for r in rows]\n    server_ms = [r[1] for r in rows]\n\n    # Throughput: replay the same payloads from `concurrency` threads sharing one client.\n    t0 = time.perf_counter()\n    with ThreadPoolExecutor(args.concurrency) as ex:\n        list(ex.map(call, payloads[args.warmup :]))\n    thr = len(payloads[args.warmup :]) / (time.perf_counter() - t0)\n\n    res = {\n        "mode": args.mode,\n        "n_requests": len(rows),\n        "components": health["components"],\n        "hardware": cpu_info(),\n        "client_ms": {\n            "p50": pct(client_ms, 50),\n            "p95": pct(client_ms, 95),\n            "mean": round(statistics.mean(client_ms), 2),\n        },\n        "server_model_ms": {"p50": pct(server_ms, 50), "p95": pct(server_ms, 95)},\n        "throughput_rps": {"concurrency": args.concurrency, "value": round(thr, 2)},\n        "note": "One item per request, sequential after warm-up; client_ms includes HTTP overhead on localhost.",\n    }\n    write_json(RESULTS / "latency.json", {args.mode: res})\n    write_json(RESULTS / "latency_samples.json", {args.mode: client_ms})\n    print(res)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/eval/metrics.py': '"""Metric helpers. Every number in results/ is produced through these functions."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.metrics import (\n    accuracy_score,\n    average_precision_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_fscore_support,\n    roc_auc_score,\n)\n\nfrom src.config import LABELS\n\n\ndef classification_report(y_true, y_pred, labels=LABELS) -> dict:\n    """Accuracy, macro-F1 and per-class precision/recall/F1/support."""\n    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)\n    idx = list(range(len(labels)))\n    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=idx, zero_division=0)\n    return {\n        "n": int(len(y_true)),\n        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),\n        "macro_f1": round(float(f1_score(y_true, y_pred, labels=idx, average="macro", zero_division=0)), 4),\n        "per_class": {\n            name: {\n                "precision": round(float(p[i]), 4),\n                "recall": round(float(r[i]), 4),\n                "f1": round(float(f[i]), 4),\n                "support": int(s[i]),\n            }\n            for i, name in enumerate(labels)\n        },\n        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=idx).tolist(),\n    }\n\n\ndef binary_report(y_true, p_pos, threshold: float = 0.5) -> dict:\n    """AUROC, average precision, and F1/macro-F1/accuracy at a probability threshold."""\n    y_true, p_pos = np.asarray(y_true).astype(int), np.asarray(p_pos, dtype=float)\n    y_hat = (p_pos >= threshold).astype(int)\n    out = {\n        "n": int(len(y_true)),\n        "positive_rate": round(float(y_true.mean()), 4),\n        "threshold": round(float(threshold), 4),\n        "accuracy": round(float(accuracy_score(y_true, y_hat)), 4),\n        "f1_harmful": round(float(f1_score(y_true, y_hat, zero_division=0)), 4),\n        "macro_f1": round(float(f1_score(y_true, y_hat, average="macro", zero_division=0)), 4),\n    }\n    if len(np.unique(y_true)) == 2:\n        out["auroc"] = round(float(roc_auc_score(y_true, p_pos)), 4)\n        out["average_precision"] = round(float(average_precision_score(y_true, p_pos)), 4)\n    return out\n\n\ndef harmful_vs_rest(y3) -> np.ndarray:\n    """Collapse 3-class labels to 1 = Harmful, 0 = Safe or Neutral."""\n    return (np.asarray(y3) == LABELS.index("Harmful")).astype(int)\n\n\ndef toxic_vs_safe(y3) -> np.ndarray:\n    """Collapse 3-class labels to 1 = Neutral or Harmful, 0 = Safe."""\n    return (np.asarray(y3) != LABELS.index("Safe")).astype(int)\n', 'src/eval/plots.py': '"""Figures for the README. All plots read from results/ or data/processed/ only.\n\nUsage:\n    python -m src.eval.plots      # regenerate every plot whose inputs exist\n"""\n\nfrom __future__ import annotations\n\nimport json\n\nimport matplotlib\n\n# headless backend; it has to be selected before pyplot is imported\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt  # noqa: E402\nimport numpy as np  # noqa: E402\nfrom sklearn.metrics import precision_recall_curve  # noqa: E402\n\nfrom src.config import LABELS, RESULTS, read_json  # noqa: E402\n\nPALETTE = {\n    "text_only": "#4C78A8",\n    "image_only": "#F58518",\n    "image_plus_weapon": "#E45756",\n    "late_fusion": "#54A24B",\n    "clip_linear_probe": "#B279A2",\n    "late_fusion_plus_clip": "#9D755D",\n}\n# the last model present in text_eval.json is the deployed one, and that is the one plotted\nTEXT_MODEL_ORDER = ["vader", "tfidf_logreg", "roberta", "distilroberta"]\n\n\ndef plot_confusion_text() -> str | None:\n    """Row-normalised Jigsaw test confusion matrix. Returns the PNG path, or None if there is nothing to plot."""\n    p = RESULTS / "text_eval.json"\n    if not p.exists():\n        return None\n    ev = read_json(p)\n    name = next((m for m in reversed(TEXT_MODEL_ORDER) if m in ev), None)\n    if name is None:\n        return None\n    cm = np.array(ev[name]["test"]["confusion_matrix"], dtype=float)\n    norm = cm / cm.sum(1, keepdims=True)\n    fig, ax = plt.subplots(figsize=(4.6, 4))\n    ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)\n    for i in range(3):\n        for j in range(3):\n            ax.text(\n                j,\n                i,\n                f"{norm[i, j]:.2f}\\n({int(cm[i, j])})",\n                ha="center",\n                va="center",\n                color="white" if norm[i, j] > 0.5 else "black",\n                fontsize=8,\n            )\n    ax.set_xticks(range(3), LABELS)\n    ax.set_yticks(range(3), LABELS)\n    ax.set_xlabel("Predicted")\n    ax.set_ylabel("True")\n    ax.set_title(f"Jigsaw test, {name} (row-normalised)", fontsize=10)\n    fig.tight_layout()\n    out = RESULTS / "confusion_text.png"\n    fig.savefig(out, dpi=150)\n    plt.close(fig)\n    return str(out)\n\n\ndef plot_fusion(curves: dict, out_dir=None) -> None:\n    """AUROC bar chart and Harmful precision-recall curves. curves: {variant: (y_true, p)} on the test split."""\n    out_dir = out_dir or RESULTS\n    from sklearn.metrics import roc_auc_score\n\n    fig, ax = plt.subplots(figsize=(6, 3.6))\n    names = list(curves)\n    aucs = [roc_auc_score(*curves[n]) for n in names]\n    bars = ax.barh(names, aucs, color=[PALETTE.get(n, "#999") for n in names])\n    for b, a in zip(bars, aucs):\n        ax.text(a + 0.005, b.get_y() + b.get_height() / 2, f"{a:.3f}", va="center", fontsize=8)\n    ax.axvline(0.5, color="#888", lw=0.8, ls="--")\n    ax.set_xlim(0.45, max(aucs) + 0.06)\n    ax.set_xlabel("AUROC (Hateful Memes test)")\n    ax.invert_yaxis()\n    fig.tight_layout()\n    fig.savefig(out_dir / "fusion_vs_single.png", dpi=150)\n    plt.close(fig)\n\n    fig, ax = plt.subplots(figsize=(5, 4))\n    for n, (y, p) in curves.items():\n        pr, rc, _ = precision_recall_curve(y, p)\n        ax.plot(rc, pr, label=n, color=PALETTE.get(n), lw=2 if n == "late_fusion" else 1.2)\n    ax.axvline(0.9, color="#888", lw=0.8, ls="--")\n    ax.text(0.905, 0.95, "90% recall", fontsize=7, color="#555")\n    ax.set_xlabel("Harmful recall")\n    ax.set_ylabel("Harmful precision")\n    ax.set_title("Precision-recall, Hateful Memes test", fontsize=10)\n    ax.legend(fontsize=7)\n    fig.tight_layout()\n    fig.savefig(out_dir / "pr_curve_harmful.png", dpi=150)\n    plt.close(fig)\n\n\ndef plot_latency(samples: dict[str, list[float]]) -> None:\n    """samples: {mode: [ms, ...]} from results/latency_samples.json, one panel per mode."""\n    modes = [m for m in ("text", "fusion") if m in samples]\n    fig, axes = plt.subplots(1, len(modes), figsize=(5 * len(modes), 3.2), squeeze=False)\n    titles = {"text": "text only (ONNX INT8)", "fusion": "text + image (late fusion)"}\n    for ax, mode in zip(axes[0], modes):\n        s = np.asarray(samples[mode])\n        ax.hist(s, bins=40, color="#4C78A8")\n        for q, ls in ((50, "-"), (95, "--")):\n            v = np.percentile(s, q)\n            ax.axvline(v, color="#E45756", ls=ls, lw=1)\n            ax.text(v, ax.get_ylim()[1] * 0.9, f" p{q} {v:.0f} ms", fontsize=7, color="#E45756")\n        ax.set_xlabel("Latency per request (ms)")\n        ax.set_ylabel("Requests")\n        ax.set_title(f"/moderate, {titles.get(mode, mode)}, CPU", fontsize=10)\n    fig.tight_layout()\n    fig.savefig(RESULTS / "latency_hist.png", dpi=150)\n    plt.close(fig)\n\n\ndef main() -> None:\n    made = [plot_confusion_text()]\n    lat = RESULTS / "latency_samples.json"\n    if lat.exists():\n        d = json.loads(lat.read_text())\n        plot_latency({k: v for k, v in d.items() if isinstance(v, list)})\n        made.append("latency_hist.png")\n    print("plots:", [m for m in made if m])\n\n\nif __name__ == "__main__":\n    main()\n', 'src/eval/triage.py': '"""Triage metric: how much manual review the model could remove.\n\nDefinition (also in the README):\n    auto_cleared_pct = share of items with p_harmful < t_clear,\nwhere t_clear is chosen on validation so that Harmful recall >= 90%.\nAuto-cleared items are labelled Safe and skip manual review, so auto_cleared_pct is\nthe estimated reduction in manual-review volume versus reviewing every item.\nWe always report the cost next to it: the share of truly harmful items that were\nauto-cleared (missed) and the harmful rate inside the cleared pile.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef triage_report(y, p, t_clear: float) -> dict:\n    """Share auto-cleared at p < t_clear and what it costs. y is binary (1 = Harmful)."""\n    y, p = np.asarray(y).astype(int), np.asarray(p, float)\n    cleared = p < t_clear\n    n_pos = int((y == 1).sum())\n    missed = int((cleared & (y == 1)).sum())\n    return {\n        "n_items": int(len(y)),\n        "t_clear": round(float(t_clear), 6),\n        "auto_cleared_pct": round(100 * float(cleared.mean()), 2),\n        "sent_to_review_or_flagged_pct": round(100 * float((~cleared).mean()), 2),\n        "harmful_recall": round(1 - missed / n_pos, 4) if n_pos else None,\n        "harmful_missed": missed,\n        "harmful_rate_in_cleared_pct": round(100 * missed / int(cleared.sum()), 2) if cleared.any() else None,\n        "base_harmful_rate_pct": round(100 * n_pos / len(y), 2),\n    }\n', 'src/fusion/__init__.py': '', 'src/fusion/calibrate.py': '"""Expected calibration error and temperature scaling."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom scipy.optimize import minimize_scalar\nfrom scipy.special import expit, log_softmax, logit\n\n\ndef ece_binary(y, p, n_bins: int = 15) -> float:\n    """ECE for P(Harmful): bin by predicted probability, average |accuracy - confidence|."""\n    y, p = np.asarray(y).astype(int), np.clip(np.asarray(p, float), 0, 1)\n    bins = np.linspace(0, 1, n_bins + 1)\n    idx = np.clip(np.digitize(p, bins[1:-1]), 0, n_bins - 1)\n    ece = 0.0\n    for b in range(n_bins):\n        m = idx == b\n        if m.any():\n            ece += m.mean() * abs(y[m].mean() - p[m].mean())\n    return float(ece)\n\n\ndef ece_multiclass(y, probs, n_bins: int = 15) -> float:\n    """Top-label ECE for multiclass probabilities (n, k)."""\n    y, probs = np.asarray(y), np.asarray(probs, float)\n    conf = probs.max(1)\n    correct = (probs.argmax(1) == y).astype(float)\n    bins = np.linspace(0, 1, n_bins + 1)\n    idx = np.clip(np.digitize(conf, bins[1:-1]), 0, n_bins - 1)\n    return float(\n        sum(\n            (idx == b).mean() * abs(correct[idx == b].mean() - conf[idx == b].mean())\n            for b in range(n_bins)\n            if (idx == b).any()\n        )\n    )\n\n\ndef fit_temperature_binary(y, p) -> float:\n    """Temperature T minimising NLL of sigmoid(logit(p) / T)."""\n    y = np.asarray(y).astype(float)\n    z = logit(np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6))\n\n    def nll(t):\n        q = np.clip(expit(z / t), 1e-9, 1 - 1e-9)\n        return -np.mean(y * np.log(q) + (1 - y) * np.log(1 - q))\n\n    return float(minimize_scalar(nll, bounds=(0.05, 20.0), method="bounded").x)\n\n\ndef apply_temperature_binary(p, t: float) -> np.ndarray:\n    """Rescale probabilities by temperature t in logit space."""\n    z = logit(np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6))\n    return expit(z / t)\n\n\ndef fit_temperature_logits(y, logits) -> float:\n    """Temperature for multiclass logits (n, k)."""\n    y, logits = np.asarray(y), np.asarray(logits, float)\n\n    def nll(t):\n        return -np.mean(log_softmax(logits / t, axis=1)[np.arange(len(y)), y])\n\n    return float(minimize_scalar(nll, bounds=(0.05, 20.0), method="bounded").x)\n\n\ndef calibration_report(y_val, p_val, y_test, p_test) -> dict:\n    """Fit T on validation, report ECE before/after on validation and test."""\n    t = fit_temperature_binary(y_val, p_val)\n    return {\n        "temperature": round(t, 4),\n        "val": {\n            "ece_before": round(ece_binary(y_val, p_val), 4),\n            "ece_after": round(ece_binary(y_val, apply_temperature_binary(p_val, t)), 4),\n        },\n        "test": {\n            "ece_before": round(ece_binary(y_test, p_test), 4),\n            "ece_after": round(ece_binary(y_test, apply_temperature_binary(p_test, t)), 4),\n        },\n    }\n', 'src/fusion/clip_probe.py': '"""Optional stronger comparison: CLIP (ViT-L/14) image + text embeddings with a linear probe. GPU helps.\n\nUsage:\n    python -m src.fusion.clip_probe [--memes-root PATH] [--model openai/clip-vit-large-patch14]\nTrains logistic regression on branch_train embeddings; writes data/processed/meme_scores_clip.csv.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegressionCV\n\nfrom src.config import DATA_PROCESSED, RESULTS, SEED, set_seed, write_json\nfrom src.image.memes import load_memes\n\n\ndef embed(df: pd.DataFrame, model, proc, device, batch: int = 64) -> np.ndarray:\n    """Per row: L2-normalised image and text embeddings and their elementwise product, concatenated."""\n    import torch\n    from PIL import Image\n\n    out = []\n    with torch.inference_mode():\n        for i in range(0, len(df), batch):\n            chunk = df.iloc[i : i + batch]\n            ims = [Image.open(p).convert("RGB") for p in chunk["img_path"]]\n            enc = proc(\n                text=chunk["text"].astype(str).tolist(),\n                images=ims,\n                return_tensors="pt",\n                padding=True,\n                truncation=True,\n                max_length=77,  # CLIP text context length\n            ).to(device)\n            o = model(**enc)\n            im = torch.nn.functional.normalize(o.image_embeds, dim=-1)\n            tx = torch.nn.functional.normalize(o.text_embeds, dim=-1)\n            out.append(torch.cat([im, tx, im * tx], -1).float().cpu().numpy())\n    return np.concatenate(out)\n\n\ndef main() -> None:\n    import torch\n    from transformers import CLIPModel, CLIPProcessor\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--memes-root")\n    ap.add_argument("--model", default="openai/clip-vit-large-patch14")\n    args = ap.parse_args()\n    set_seed()\n    splits, _ = load_memes(args.memes_root)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    model = CLIPModel.from_pretrained(args.model).to(dev).eval()\n    proc = CLIPProcessor.from_pretrained(args.model)\n    X = {k: embed(v, model, proc, dev) for k, v in splits.items()}\n    clf = LogisticRegressionCV(\n        Cs=[0.01, 0.03, 0.1, 0.3, 1, 3],\n        cv=5,\n        scoring="roc_auc",\n        max_iter=3000,\n        class_weight="balanced",\n        random_state=SEED,\n    )\n    clf.fit(X["branch_train"], splits["branch_train"]["label"])\n    rows = [\n        pd.DataFrame(\n            {"id": splits[s]["id"], "split": s, "label": splits[s]["label"], "p_clip": clf.predict_proba(X[s])[:, 1]}\n        )\n        for s in ("fusion_train", "test")\n    ]\n    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)\n    pd.concat(rows).to_csv(DATA_PROCESSED / "meme_scores_clip.csv", index=False)\n    write_json(RESULTS / "training_log.json", {"clip_probe": {"model": args.model, "C": float(clf.C_[0])}})\n\n\nif __name__ == "__main__":\n    main()\n', 'src/fusion/features.py': '"""Build fusion feature matrices from per-branch score files."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.config import DATA_PROCESSED\n\nBRANCH_FILES = ["meme_scores_text.csv", "meme_scores_image.csv", "meme_scores_weapon.csv", "meme_scores_clip.csv"]\n# columns in [0, 1] that are logit-transformed before fitting\nPROB_COLS = {"p_text", "p_image", "p_text_jigsaw", "p_clip", "weapon_max_conf"}\n\n# Feature sets compared in results/fusion_eval.json. Columns missing from the data are skipped.\nVARIANTS: dict[str, list[str]] = {\n    "text_only": ["p_text"],\n    "image_only": ["p_image"],\n    "image_plus_weapon": ["p_image", "weapon_max_conf", "weapon_present"],\n    "late_fusion": ["p_text", "p_image", "weapon_max_conf", "weapon_present", "p_text_jigsaw", "text_x_image"],\n    "clip_linear_probe": ["p_clip"],\n    "text_plus_clip": ["p_text", "p_text_jigsaw", "p_clip"],\n    "late_fusion_plus_clip": [\n        "p_text",\n        "p_image",\n        "weapon_max_conf",\n        "weapon_present",\n        "p_text_jigsaw",\n        "text_x_image",\n        "p_clip",\n    ],\n}\nDEPLOYED = "late_fusion"\n\n\ndef load_branch_scores(processed: Path = DATA_PROCESSED) -> pd.DataFrame:\n    """Inner-join every available branch file on (id, split, label), keeping items scored by all of them."""\n    df = None\n    for name in BRANCH_FILES:\n        p = processed / name\n        if not p.exists():\n            continue\n        part = pd.read_csv(p)\n        df = part if df is None else df.merge(part, on=["id", "split", "label"], how="inner")\n    if df is None:\n        raise FileNotFoundError(f"no meme_scores_*.csv files in {processed}")\n    return df\n\n\ndef _logit(p: np.ndarray) -> np.ndarray:\n    p = np.clip(p, 1e-4, 1 - 1e-4)\n    return np.log(p / (1 - p))\n\n\ndef make_features(df: pd.DataFrame, cols: list[str]) -> tuple[np.ndarray, list[str]]:\n    """Return X and the column names actually used. Probabilities are logit-transformed."""\n    feats, used = [], []\n    for c in cols:\n        if c == "text_x_image":\n            # interaction term: product of the text and image logits, scaled down by 10\n            if {"p_text", "p_image"} <= set(df.columns):\n                feats.append(_logit(df["p_text"].to_numpy()) * _logit(df["p_image"].to_numpy()) / 10)\n                used.append(c)\n            continue\n        if c not in df.columns:\n            continue\n        v = df[c].to_numpy(dtype=float)\n        feats.append(_logit(v) if c in PROB_COLS else v)\n        used.append(c)\n    if not feats:\n        return np.zeros((len(df), 0)), []\n    return np.column_stack(feats), used\n\n\ndef features_from_scores(scores: dict, cols: list[str]) -> np.ndarray:\n    """Single-item version used by the API. `scores` holds the same keys as the columns."""\n    return make_features(pd.DataFrame([scores]), cols)[0]\n', 'src/fusion/threshold.py': '"""Threshold selection for the Harmful score.\n\nTwo thresholds turn one probability p_harmful into three actions:\n\n    p <  t_clear            -> Safe     (auto-cleared, no human review)\n    t_clear <= p < t_flag   -> Neutral  (sent to manual review)\n    p >= t_flag             -> Harmful  (auto-flagged, high precision)\n\nt_clear is the largest threshold that keeps Harmful recall >= TARGET_RECALL on validation.\nt_flag is the smallest threshold that reaches the target precision on validation.\nBoth are chosen on validation and applied unchanged to test.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\n\nimport numpy as np\n\nfrom src.config import TARGET_RECALL\n\n\n@dataclass\nclass Thresholds:\n    """The two cut-offs on p_harmful and the targets they were fitted for."""\n\n    t_clear: float\n    t_flag: float\n    target_recall: float\n    target_precision: float\n\n    def to_dict(self) -> dict:\n        return {k: round(float(v), 6) for k, v in asdict(self).items()}\n\n\ndef recall_precision_at(y, p, t: float) -> tuple[float, float, float]:\n    """Harmful recall, precision and flagged share when flagging p >= t."""\n    y, p = np.asarray(y).astype(int), np.asarray(p, float)\n    flagged = p >= t\n    tp = int((flagged & (y == 1)).sum())\n    pos = int((y == 1).sum())\n    recall = tp / pos if pos else float("nan")\n    precision = tp / int(flagged.sum()) if flagged.any() else float("nan")\n    return recall, precision, float(flagged.mean())\n\n\ndef threshold_for_recall(y, p, target: float = TARGET_RECALL) -> float:\n    """Largest threshold t such that recall(p >= t) >= target.\n\n    With positive scores sorted in descending order, flagging everything >= the k-th one\n    gives recall k / pos, so k = ceil(target * pos).\n    """\n    y, p = np.asarray(y).astype(int), np.asarray(p, float)\n    pos_scores = np.sort(p[y == 1])[::-1]\n    if len(pos_scores) == 0:\n        raise ValueError("no positive examples to set a recall threshold")\n    k = int(np.ceil(target * len(pos_scores)))\n    k = min(max(k, 1), len(pos_scores))\n    return float(pos_scores[k - 1])\n\n\ndef threshold_for_precision(y, p, target: float = 0.8, floor: float = 0.5) -> float:\n    """Smallest threshold with precision >= target over at least 5 flagged items.\n\n    Falls back to max(floor, highest score) when no threshold qualifies.\n    """\n    y, p = np.asarray(y).astype(int), np.asarray(p, float)\n    candidates = np.unique(p)\n    for t in candidates:  # ascending\n        _, prec, _ = recall_precision_at(y, p, t)\n        if not np.isnan(prec) and prec >= target and (p >= t).sum() >= 5:\n            return float(t)\n    return float(max(floor, candidates.max()))\n\n\ndef fit_thresholds(y_val, p_val, target_recall: float = TARGET_RECALL, target_precision: float = 0.8) -> Thresholds:\n    """Fit t_clear for the recall target and t_flag for the precision target; t_flag is never below t_clear."""\n    t_clear = threshold_for_recall(y_val, p_val, target_recall)\n    t_flag = max(threshold_for_precision(y_val, p_val, target_precision), t_clear)\n    return Thresholds(t_clear, t_flag, target_recall, target_precision)\n\n\ndef decide(p, th: Thresholds) -> np.ndarray:\n    """Vectorised 3-way decision: 0 Safe, 1 Neutral (review), 2 Harmful."""\n    p = np.asarray(p, float)\n    out = np.ones(len(p), dtype=int)\n    out[p < th.t_clear] = 0\n    out[p >= th.t_flag] = 2\n    return out\n\n\ndef threshold_report(y, p, th: Thresholds) -> dict:\n    """Harmful recall, precision and flagged share at t_clear and at t_flag."""\n    r, pr, flagged = recall_precision_at(y, p, th.t_clear)\n    rf, prf, flagged_f = recall_precision_at(y, p, th.t_flag)\n    return {\n        "thresholds": th.to_dict(),\n        "at_t_clear": {\n            "harmful_recall": round(r, 4),\n            "harmful_precision": round(pr, 4),\n            "flagged_share": round(flagged, 4),\n        },\n        "at_t_flag": {\n            "harmful_recall": round(rf, 4),\n            "harmful_precision": round(prf, 4),\n            "flagged_share": round(flagged_f, 4),\n        },\n    }\n', 'src/fusion/train_fusion.py': '"""Late fusion on Hateful Memes, with threshold tuning, triage and calibration. CPU, seconds.\n\nUsage:\n    python -m src.fusion.train_fusion [--memes-root PATH]\n\nInput: data/processed/meme_scores_*.csv (from the GPU notebook).\nFor every feature set in features.VARIANTS:\n    * standardised logistic regression trained on fusion_train (5-fold out-of-fold scores for tuning)\n    * evaluated on test: AUROC, average precision, macro-F1 at the OOF-best-F1 threshold\nFor the headline variant (best out-of-fold AUROC):\n    * t_clear (Harmful recall >= 90% on OOF) and t_flag (precision >= 0.8) -> results/threshold.json\n    * triage metric on test -> results/triage.json\n    * failure examples -> results/examples.md\nTemperature scaling ECE for the headline and deployed variants -> results/calibration.json (keys fusion_*).\nModel bundle for the API (deployed variant late_fusion, with its own thresholds): models/fusion_lr.joblib\n"""\n\nfrom __future__ import annotations\n\nimport argparse\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import f1_score\nfrom sklearn.model_selection import StratifiedKFold, cross_val_predict\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom src.config import DATA_PROCESSED, MODELS, RESULTS, SEED, TARGET_RECALL, set_seed, write_json\nfrom src.eval.metrics import binary_report\nfrom src.eval.triage import triage_report\nfrom src.fusion.calibrate import calibration_report, fit_temperature_binary\nfrom src.fusion.features import DEPLOYED, VARIANTS, load_branch_scores, make_features\nfrom src.fusion.threshold import decide, fit_thresholds, threshold_for_recall, threshold_report\n\nSCORES_DIR = DATA_PROCESSED  # module-level so tests can point it at fixtures\n\n\ndef make_model():\n    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))\n\n\ndef best_f1_threshold(y, p) -> float:\n    """Threshold maximising macro-F1 over the 2nd to 98th percentiles of p (97-point grid)."""\n    grid = np.unique(np.quantile(p, np.linspace(0.02, 0.98, 97)))\n    scores = [f1_score(y, p >= t, average="macro") for t in grid]\n    return float(grid[int(np.argmax(scores))])\n\n\ndef fit_variant(df: pd.DataFrame, cols: list[str]):\n    """Fit one feature set on fusion_train and score test. Returns None if none of its columns exist."""\n    tr, te = df[df.split == "fusion_train"], df[df.split == "test"]\n    Xtr, used = make_features(tr, cols)\n    if not used:\n        return None\n    Xte, _ = make_features(te, cols)\n    ytr, yte = tr["label"].to_numpy(), te["label"].to_numpy()\n    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)\n    p_oof = cross_val_predict(make_model(), Xtr, ytr, cv=cv, method="predict_proba")[:, 1]\n    model = make_model().fit(Xtr, ytr)\n    p_te = model.predict_proba(Xte)[:, 1]\n    t = best_f1_threshold(ytr, p_oof)\n    return {\n        "features": used,\n        "model": model,\n        "p_oof": p_oof,\n        "p_test": p_te,\n        "y_oof": ytr,\n        "y_test": yte,\n        "report": {"features": used, "oof": binary_report(ytr, p_oof, t), "test": binary_report(yte, p_te, t)},\n    }\n\n\ndef write_examples(df_test: pd.DataFrame, p: np.ndarray, th, captions: dict | None, model: str, k: int = 5) -> None:\n    """Write the k most confident misses and false alarms on test to results/examples.md."""\n    d = df_test.copy()\n    d["p_harmful"] = p\n    d["decision"] = np.array(["Safe", "Neutral", "Harmful"])[decide(p, th)]\n    fn = d[(d.label == 1) & (d.decision == "Safe")].sort_values("p_harmful").head(k)\n    fp = d[(d.label == 0) & (d.decision == "Harmful")].sort_values("p_harmful", ascending=False).head(k)\n    cols = [c for c in ("p_text", "p_text_jigsaw", "p_image", "p_clip", "weapon_max_conf", "p_harmful") if c in d]\n    lines = [\n        f"# Failure examples (Hateful Memes test split, {model} model)",\n        "",\n        f"Thresholds: t_clear = {th.t_clear:.4f}, t_flag = {th.t_flag:.4f}. Captions are the dataset\'s "\n        "overlaid text; images are not reproduced here.",\n        "",\n    ]\n    for title, sub in (\n        ("Missed harmful memes (auto-cleared as Safe)", fn),\n        ("False alarms (auto-flagged Harmful)", fp),\n    ):\n        lines += [f"## {title}", "", "| id | caption | " + " | ".join(cols) + " |", "|---|---|" + "---|" * len(cols)]\n        for _, r in sub.iterrows():\n            cap = (captions or {}).get(r["id"], "")\n            cap = str(cap).replace("|", "/")[:90]\n            lines.append(f"| {r[\'id\']} | {cap} | " + " | ".join(f"{r[c]:.3f}" for c in cols) + " |")\n        lines += ["", "Why it fails: _fill in after reading the examples (see HANDOVER step 6)._", ""]\n    (RESULTS / "examples.md").write_text("\\n".join(lines), encoding="utf-8")\n\n\ndef main(argv: list[str] | None = None) -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--memes-root")\n    args = ap.parse_args(argv)\n    set_seed()\n    df = load_branch_scores(SCORES_DIR)\n    fits, seen = {}, set()\n    for name, cols in VARIANTS.items():\n        f = fit_variant(df, cols)\n        if f is None or tuple(f["features"]) in seen:\n            continue  # no usable columns, or missing branch files reduced it to a variant already fitted\n        seen.add(tuple(f["features"]))\n        fits[name] = f\n\n    # thresholds and temperature for the API bundle come from the deployed variant\n    dep = fits[DEPLOYED]\n    th = fit_thresholds(dep["y_oof"], dep["p_oof"], TARGET_RECALL)\n    t_temp = fit_temperature_binary(dep["y_oof"], dep["p_oof"])\n\n    # Headline model: best out-of-fold AUROC on fusion_train, never chosen on test.\n    head_name = max(fits, key=lambda k: fits[k]["report"]["oof"].get("auroc", 0))\n    head = fits[head_name]\n    th_head = fit_thresholds(head["y_oof"], head["p_oof"], TARGET_RECALL)\n\n    single = {k: fits[k]["report"]["test"].get("auroc") for k in ("text_only", "image_only") if k in fits}\n    best_single = max(single, key=single.get) if single else None\n    write_json(\n        RESULTS / "fusion_eval.json",\n        {\n            "dataset": "Hateful Memes",\n            "eval_split_note": "test = labelled split named in data_summary.json (hateful_memes.test_file)",\n            "protocol": "branches trained on branch_train; fusion LR trained on fusion_train (held out); thresholds from 5-fold OOF on fusion_train; metrics on test",\n            "variants": {k: v["report"] for k, v in fits.items()},\n            "headline": head_name,\n            "headline_rule": "highest out-of-fold AUROC on fusion_train",\n            "deployed_api": DEPLOYED,\n            "best_single_branch": best_single,\n            "fusion_minus_best_single_auroc": round(dep["report"]["test"]["auroc"] - single[best_single], 4)\n            if best_single\n            else None,\n            "headline_minus_text_only_auroc": round(head["report"]["test"]["auroc"] - single["text_only"], 4)\n            if "text_only" in single\n            else None,\n        },\n        merge=False,\n    )\n\n    per_variant = {}\n    for k, v in fits.items():\n        t_k = fit_thresholds(v["y_oof"], v["p_oof"], TARGET_RECALL)\n        tr_k = triage_report(v["y_test"], v["p_test"], t_k.t_clear)\n        per_variant[k] = {\n            "t_clear": round(t_k.t_clear, 6),\n            "oof_harmful_recall": threshold_report(v["y_oof"], v["p_oof"], t_k)["at_t_clear"]["harmful_recall"],\n            "test_harmful_recall": tr_k["harmful_recall"],\n            "test_harmful_precision": threshold_report(v["y_test"], v["p_test"], t_k)["at_t_clear"][\n                "harmful_precision"\n            ],\n            "test_auto_cleared_pct": tr_k["auto_cleared_pct"],\n        }\n    rep = threshold_report(head["y_test"], head["p_test"], th_head)\n    rep["variant"] = head_name\n    rep["val_oof"] = threshold_report(head["y_oof"], head["p_oof"], th_head)\n    rep["per_variant"] = per_variant\n    rep["note"] = (\n        f"Thresholds chosen on fusion_train out-of-fold scores (target Harmful recall {TARGET_RECALL}); applied unchanged to test."\n    )\n    write_json(RESULTS / "threshold.json", rep, merge=False)\n    tri = triage_report(head["y_test"], head["p_test"], th_head.t_clear)\n    tri["variant"] = head_name\n    # Reference only (uses test labels, never used for any decision): the share that could be cleared if the\n    # threshold were set on test itself to hit exactly 90% recall. It sizes the cost of the val-to-test shift.\n    t_oracle = threshold_for_recall(head["y_test"], head["p_test"], TARGET_RECALL)\n    tri["reference_test_oracle_at_target_recall"] = {\n        "auto_cleared_pct": triage_report(head["y_test"], head["p_test"], t_oracle)["auto_cleared_pct"],\n        "note": "threshold picked on test labels; for context only, not a result of the method",\n    }\n    tri["deployed_api"] = {DEPLOYED: per_variant[DEPLOYED]}\n    tri["definition"] = (\n        "auto_cleared_pct = share of test items with p_harmful < t_clear, where t_clear keeps "\n        "Harmful recall >= 90% on validation (out-of-fold). These items skip manual review."\n    )\n    write_json(RESULTS / "triage.json", tri, merge=False)\n    write_json(\n        RESULTS / "calibration.json",\n        {\n            f"fusion_{head_name}": calibration_report(head["y_oof"], head["p_oof"], head["y_test"], head["p_test"]),\n            f"fusion_{DEPLOYED}": calibration_report(dep["y_oof"], dep["p_oof"], dep["y_test"], dep["p_test"]),\n        },\n    )\n\n    MODELS.mkdir(parents=True, exist_ok=True)\n    joblib.dump(\n        {"model": dep["model"], "features": dep["features"], "thresholds": th.to_dict(), "temperature": t_temp},\n        MODELS / "fusion_lr.joblib",\n    )\n\n    captions = None\n    try:\n        from src.image.memes import load_memes\n\n        s, _ = load_memes(args.memes_root)\n        captions = dict(zip(s["test"]["id"].astype(int), s["test"]["text"]))\n    except FileNotFoundError:\n        pass\n    te = df[df.split == "test"].reset_index(drop=True)\n    write_examples(te, head["p_test"], th_head, captions, head_name)\n\n    from src.eval.plots import plot_fusion\n\n    plot_fusion({k: (v["y_test"], v["p_test"]) for k, v in fits.items()}, out_dir=RESULTS)\n    for k, v in fits.items():\n        print(f"{k:>22}: test AUROC {v[\'report\'][\'test\'].get(\'auroc\')}  macro-F1 {v[\'report\'][\'test\'][\'macro_f1\']}")\n    print("headline:", head_name, "triage:", tri["auto_cleared_pct"], "% auto-cleared at recall", tri["harmful_recall"])\n\n\nif __name__ == "__main__":\n    main()\n', 'src/image/__init__.py': '', 'src/image/infer_image.py': '"""EfficientNet-B0 image scorer: P(harmful | image)."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom src.config import MODELS\n\nIMG_SIZE = 256\n# ImageNet normalisation, matching the pretrained timm weights\nMEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)\n\n\ndef build_transforms(train: bool):\n    from torchvision import transforms as T\n\n    if train:\n        return T.Compose(\n            [\n                T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),\n                T.RandomHorizontalFlip(),\n                T.ColorJitter(0.2, 0.2, 0.2),\n                T.ToTensor(),\n                T.Normalize(MEAN, STD),\n            ]\n        )\n    return T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])\n\n\ndef build_model(pretrained: bool = True):\n    """EfficientNet-B0 with a single logit output (sigmoid gives P(harmful))."""\n    import timm\n\n    return timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=1)\n\n\nclass ImageScorer:\n    def __init__(self, weights: Path = MODELS / "effnet_b0_memes.pt", device: str | None = None):\n        import torch\n\n        self.torch = torch\n        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        self.model = build_model(pretrained=False)\n        self.model.load_state_dict(torch.load(weights, map_location="cpu"))\n        self.model.to(self.device).eval()\n        self.tf = build_transforms(train=False)\n\n    def predict(self, images: list, batch: int = 32) -> np.ndarray:\n        """images: list of PIL images. Returns P(harmful) per image."""\n        out = []\n        with self.torch.inference_mode():\n            for i in range(0, len(images), batch):\n                x = self.torch.stack([self.tf(im.convert("RGB")) for im in images[i : i + batch]]).to(self.device)\n                out.append(self.torch.sigmoid(self.model(x).squeeze(-1)).float().cpu().numpy())\n        return np.concatenate(out) if out else np.zeros(0)\n', 'src/image/infer_yolo.py': '"""Weapon detector features for fusion and for the API explanation.\n\nUsage (scores every meme in fusion_train and test):\n    python -m src.image.infer_yolo [--memes-root PATH]\nWrites data/processed/meme_scores_weapon.csv.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.config import DATA_PROCESSED, MODELS\n\nCONF_PRESENT = 0.35  # a box above this confidence counts as "weapon present"\n\n\nclass WeaponDetector:\n    def __init__(self, weights: Path = MODELS / "yolov8s_weapons.pt", device: str | None = None):\n        from ultralytics import YOLO, settings\n\n        settings.update({"sync": False})  # no usage analytics\n        self.model = YOLO(str(weights))\n        self.device = device\n\n    def features(self, images: list, batch: int = 32) -> list[dict]:\n        """images: PIL images or paths. Returns one feature dict per image."""\n        feats = []\n        for i in range(0, len(images), batch):\n            # low conf floor so weapon_max_conf still carries signal below CONF_PRESENT\n            for r in self.model.predict(images[i : i + batch], conf=0.05, verbose=False, device=self.device):\n                conf = r.boxes.conf.cpu().numpy() if len(r.boxes) else np.zeros(0)\n                cls = r.boxes.cls.cpu().numpy().astype(int) if len(r.boxes) else np.zeros(0, int)\n                top = int(conf.argmax()) if len(conf) else -1\n                feats.append(\n                    {\n                        "weapon_max_conf": float(conf.max()) if len(conf) else 0.0,\n                        "weapon_present": int(bool(len(conf)) and conf.max() >= CONF_PRESENT),\n                        "weapon_count": int((conf >= CONF_PRESENT).sum()),\n                        "weapon_class": r.names[int(cls[top])] if top >= 0 else None,\n                    }\n                )\n        return feats\n\n\ndef main() -> None:\n    from src.image.memes import load_memes\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--memes-root")\n    args = ap.parse_args()\n    splits, _ = load_memes(args.memes_root)\n    det = WeaponDetector()\n    rows = []\n    for split in ("fusion_train", "test"):\n        df = splits[split]\n        f = pd.DataFrame(det.features(df["img_path"].tolist()))\n        rows.append(pd.concat([pd.DataFrame({"id": df["id"], "split": split, "label": df["label"]}), f], axis=1))\n    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)\n    pd.concat(rows).to_csv(DATA_PROCESSED / "meme_scores_weapon.csv", index=False)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/image/memes.py': '"""Hateful Memes loader and the split protocol used for fusion.\n\nExpected layout (the Kaggle mirror and the official release both match):\n    <root>/train.jsonl, <root>/dev.jsonl or dev_seen.jsonl, optional test_seen.jsonl, <root>/img/*.png\nEach JSON line: {"id": 42953, "img": "img/42953.png", "label": 0/1, "text": "..."}.\n\nSplits (seeded, stratified):\n    branch_train  80% of train.jsonl  -> trains the text, image and CLIP branches\n    fusion_train  20% of train.jsonl  -> held out from every branch; trains the fusion model\n                                         and picks thresholds (out-of-fold)\n    test          labelled test_seen.jsonl if present, else dev_seen.jsonl / dev.jsonl\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nfrom pathlib import Path\n\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\n\nfrom src.config import SEED\n\nDEFAULT_CANDIDATES = [\n    Path("/kaggle/input/facebook-hateful-meme-dataset/data"),\n    Path("/kaggle/input/hateful-memes/data"),\n    Path("/kaggle/input/hateful-memes"),\n    Path("data/raw/hateful_memes"),\n]\n\n\ndef find_root(explicit: str | Path | None = None) -> Path:\n    """First directory containing train.jsonl: explicit path, MM_MEMES_ROOT, known locations, then /kaggle/input."""\n    cands = [Path(explicit)] if explicit else []\n    if os.environ.get("MM_MEMES_ROOT"):\n        env_root = Path(os.environ["MM_MEMES_ROOT"])\n        cands += [env_root] + [p.parent for p in env_root.glob("**/train.jsonl")]\n    cands += DEFAULT_CANDIDATES\n    cands += [p.parent for p in Path("/kaggle/input").glob("**/train.jsonl")] if Path("/kaggle/input").exists() else []\n    for c in cands:\n        if (c / "train.jsonl").exists():\n            return c\n    raise FileNotFoundError("Hateful Memes not found; attach the dataset or pass --memes-root")\n\n\ndef _read_jsonl(path: Path) -> pd.DataFrame:\n    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]\n    return pd.DataFrame(rows)\n\n\ndef load_memes(root: str | Path | None = None, fusion_frac: float = 0.2, seed: int = SEED) -> tuple[dict, dict]:\n    """Return (splits, meta): the three split frames and a summary for data_summary.json."""\n    root = find_root(root)\n    train = _read_jsonl(root / "train.jsonl")\n    test, test_name = None, None\n    # first fully labelled file wins; unlabelled test files are skipped\n    for name in ("test_seen.jsonl", "dev_seen.jsonl", "dev.jsonl"):\n        p = root / name\n        if p.exists():\n            df = _read_jsonl(p)\n            if "label" in df and df["label"].notna().all():\n                test, test_name = df, name\n                break\n    if test is None:\n        raise FileNotFoundError("no labelled evaluation split found")\n    for df in (train, test):\n        df["img_path"] = df["img"].map(lambda x: str(root / x))\n        df["label"] = df["label"].astype(int)\n    bt, ft = train_test_split(train, test_size=fusion_frac, random_state=seed, stratify=train["label"])\n    splits = {\n        "branch_train": bt.reset_index(drop=True),\n        "fusion_train": ft.reset_index(drop=True),\n        "test": test.reset_index(drop=True),\n    }\n    meta = {\n        "root": str(root),\n        "test_file": test_name,\n        "sizes": {k: {"n": len(v), "hateful": int(v["label"].sum())} for k, v in splits.items()},\n    }\n    return splits, meta\n', 'src/image/train_effnet.py': '"""Fine-tune EfficientNet-B0 (ImageNet-pretrained, timm) as the image-only Hateful Memes branch. GPU.\n\nUsage:\n    python -m src.image.train_effnet [--memes-root PATH] [--epochs 8]\n\nTrains on branch_train (with a 10% early-stopping slice), then scores fusion_train and test.\nWrites models/effnet_b0_memes.pt and data/processed/meme_scores_image.csv.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score\nfrom sklearn.model_selection import train_test_split\n\nfrom src.config import DATA_PROCESSED, MODELS, RESULTS, SEED, set_seed, write_json\nfrom src.image.infer_image import build_model, build_transforms\nfrom src.image.memes import load_memes\n\n\ndef main() -> None:\n    import torch\n    from PIL import Image\n    from torch.utils.data import DataLoader, Dataset\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--memes-root")\n    ap.add_argument("--epochs", type=int, default=8)\n    ap.add_argument("--batch", type=int, default=64)\n    ap.add_argument("--lr", type=float, default=3e-4)\n    ap.add_argument("--no-pretrained", action="store_true", help="random init (offline smoke tests only)")\n    args = ap.parse_args()\n    set_seed()\n    splits, meta = load_memes(args.memes_root)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n\n    class DS(Dataset):\n        def __init__(self, df, train):\n            self.df, self.tf = df.reset_index(drop=True), build_transforms(train)\n\n        def __len__(self):\n            return len(self.df)\n\n        def __getitem__(self, i):\n            r = self.df.iloc[i]\n            return self.tf(Image.open(r["img_path"]).convert("RGB")), torch.tensor(float(r["label"]))\n\n    tr, es = train_test_split(\n        splits["branch_train"], test_size=0.1, random_state=SEED, stratify=splits["branch_train"]["label"]\n    )\n    dl = DataLoader(DS(tr, True), batch_size=args.batch, shuffle=True, num_workers=2, drop_last=True)\n\n    def loader(df):\n        return DataLoader(DS(df, False), batch_size=args.batch * 2, num_workers=2)\n\n    model = build_model(pretrained=not args.no_pretrained).to(dev)\n    # negatives / positives, so both classes carry equal weight in the BCE loss\n    pos_weight = torch.tensor((tr["label"] == 0).sum() / max((tr["label"] == 1).sum(), 1), device=dev)\n    opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)\n    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=args.lr, total_steps=args.epochs * len(dl))\n    scaler = torch.amp.GradScaler("cuda", enabled=dev == "cuda")\n\n    def predict(df) -> np.ndarray:\n        model.eval()\n        out = []\n        with torch.inference_mode():\n            for x, _ in loader(df):\n                with torch.autocast(dev, enabled=dev == "cuda"):\n                    out.append(torch.sigmoid(model(x.to(dev)).squeeze(-1)).float().cpu().numpy())\n        return np.concatenate(out)\n\n    best, best_state, log = -1.0, None, []\n    t0 = time.time()\n    for ep in range(args.epochs):\n        model.train()\n        for x, y in dl:\n            x, y = x.to(dev), y.to(dev)\n            with torch.autocast(dev, enabled=dev == "cuda"):\n                loss = torch.nn.functional.binary_cross_entropy_with_logits(\n                    model(x).squeeze(-1), y, pos_weight=pos_weight\n                )\n            opt.zero_grad()\n            scaler.scale(loss).backward()\n            scaler.step(opt)\n            scaler.update()\n            sched.step()\n        auc = roc_auc_score(es["label"], predict(es))\n        log.append({"epoch": ep + 1, "early_stop_auroc": round(float(auc), 4)})\n        print(log[-1])\n        # keep the epoch with the best early-stopping AUROC\n        if auc > best:\n            best, best_state = auc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n    model.load_state_dict(best_state)\n    MODELS.mkdir(parents=True, exist_ok=True)\n    torch.save(best_state, MODELS / "effnet_b0_memes.pt")\n\n    rows = []\n    for split in ("fusion_train", "test"):\n        df = splits[split]\n        rows.append(pd.DataFrame({"id": df["id"], "split": split, "label": df["label"], "p_image": predict(df)}))\n    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)\n    pd.concat(rows).to_csv(DATA_PROCESSED / "meme_scores_image.csv", index=False)\n    write_json(\n        RESULTS / "training_log.json",\n        {\n            "effnet_b0": {\n                "epochs": log,\n                "best_early_stop_auroc": round(best, 4),\n                "train_seconds": round(time.time() - t0, 1),\n                "data": meta,\n            }\n        },\n    )\n\n\nif __name__ == "__main__":\n    main()\n', 'src/image/train_yolo.py': '"""Fine-tune YOLOv8s on an Open Images V7 weapon subset (Handgun, Rifle, Shotgun, Knife). GPU.\n\nUsage:\n    python -m src.image.train_yolo [--per-class 1000] [--epochs 40]\n\n1. Downloads the subset with FiftyOne (train / validation / test splits of Open Images V7),\n   keeping only boxes of the four classes, and exports it in YOLO format.\n2. Trains yolov8s.pt (COCO-pretrained) with a fixed seed.\n3. Evaluates on the held-out test split and writes results/detector_eval.json\n   (mAP50, mAP50-95, precision, recall, per-class mAP50-95).\nWeights: models/yolov8s_weapons.pt\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport shutil\nimport time\nfrom pathlib import Path\n\nfrom src.config import DATA_RAW, MODELS, RESULTS, SEED, set_seed, write_json\n\nCLASSES = ["Handgun", "Rifle", "Shotgun", "Knife"]\n\n\ndef build_dataset(out_dir: Path, per_class: int) -> tuple[Path, dict]:\n    import fiftyone as fo\n    import fiftyone.zoo as foz\n    from fiftyone import ViewField as F\n\n    counts = {}\n    # validation and test get a fifth of the train budget per class, at least 50\n    budget = {"train": per_class, "validation": max(per_class // 5, 50), "test": max(per_class // 5, 50)}\n    for split, yolo_split in (("train", "train"), ("validation", "val"), ("test", "test")):\n        views = []\n        for cls in CLASSES:  # per-class download keeps the classes balanced\n            ds = foz.load_zoo_dataset(\n                "open-images-v7",\n                split=split,\n                label_types=["detections"],\n                classes=[cls],\n                max_samples=budget[split],\n                seed=SEED,\n                shuffle=True,\n                dataset_name=f"weapons-{split}-{cls}",\n            )\n            views.append(ds)\n        merged = fo.Dataset(f"weapons-{split}-merged", overwrite=True)\n        for v in views:\n            merged.add_samples(v.filter_labels("ground_truth", F("label").is_in(CLASSES)))\n        merged = merged.filter_labels("ground_truth", F("label").is_in(CLASSES))\n        counts[yolo_split] = {"images": len(merged), "boxes": merged.count("ground_truth.detections")}\n        merged.export(\n            export_dir=str(out_dir),\n            dataset_type=fo.types.YOLOv5Dataset,\n            label_field="ground_truth",\n            split=yolo_split,\n            classes=CLASSES,\n        )\n    return out_dir / "dataset.yaml", counts\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--per-class", type=int, default=1000, help="train images per class before filtering")\n    ap.add_argument("--epochs", type=int, default=40)\n    ap.add_argument("--imgsz", type=int, default=640)\n    ap.add_argument("--batch", type=int, default=16)\n    ap.add_argument("--weights", default="yolov8s.pt", help="starting checkpoint (COCO-pretrained yolov8s by default)")\n    ap.add_argument("--data-yaml", type=Path, help="skip download and use an existing YOLO dataset.yaml")\n    args = ap.parse_args()\n    set_seed()\n    from ultralytics import YOLO, settings\n\n    settings.update({"sync": False})  # no usage analytics\n\n    if args.data_yaml:\n        yaml_path, counts = args.data_yaml, {"note": "pre-built dataset"}\n    else:\n        yaml_path, counts = build_dataset(DATA_RAW / "weapons_yolo", args.per_class)\n\n    t0 = time.time()\n    model = YOLO(args.weights)\n    model.train(\n        data=str(yaml_path),\n        epochs=args.epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        seed=SEED,\n        deterministic=True,\n        patience=15,\n        project=str(MODELS / "yolo_runs"),\n        name="weapons",\n        exist_ok=True,\n    )\n    train_s = time.time() - t0\n    best = Path(model.trainer.best)\n    MODELS.mkdir(parents=True, exist_ok=True)\n    shutil.copy(best, MODELS / "yolov8s_weapons.pt")\n\n    m = YOLO(str(MODELS / "yolov8s_weapons.pt")).val(\n        data=str(yaml_path),\n        split="test",\n        imgsz=args.imgsz,\n        seed=SEED,\n        project=str(MODELS / "yolo_runs"),\n        name="test_eval",\n    )\n    names = m.names if hasattr(m, "names") else dict(enumerate(CLASSES))\n    per_class = {\n        names[int(c)]: round(float(v), 4) for c, v in zip(m.box.ap_class_index, m.box.maps[m.box.ap_class_index])\n    }\n    write_json(\n        RESULTS / "detector_eval.json",\n        {\n            "model": f"{args.weights} fine-tuned",\n            "dataset": "Open Images V7 weapon subset via FiftyOne",\n            "classes": CLASSES,\n            "split_counts": counts,\n            "epochs": args.epochs,\n            "imgsz": args.imgsz,\n            "test": {\n                "mAP50": round(float(m.box.map50), 4),\n                "mAP50_95": round(float(m.box.map), 4),\n                "precision": round(float(m.box.mp), 4),\n                "recall": round(float(m.box.mr), 4),\n                "per_class_mAP50_95": per_class,\n            },\n            "train_seconds": round(train_s, 1),\n        },\n        merge=False,\n    )\n    write_json(\n        RESULTS / "data_summary.json",\n        {\n            "open_images_weapons": {\n                "source": "Open Images V7 (images CC BY 2.0 per Flickr owner; annotations CC BY 4.0)",\n                "classes": CLASSES,\n                "counts": counts,\n                "seed": SEED,\n            }\n        },\n    )\n    print("test mAP50", m.box.map50, "mAP50-95", m.box.map)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/serve/__init__.py': '', 'src/serve/app.py': '"""FastAPI moderation service.\n\nRun:\n    uvicorn src.serve.app:app --host 0.0.0.0 --port 8000\n\nEndpoints:\n    GET  /health          which components are loaded\n    POST /moderate        multipart form: text, image_url, image (file); any subset, at least one\n    POST /moderate/json   JSON body: {"text": ..., "image_url": ..., "image_b64": ...}\n\nModels load once at startup from models/. Missing components degrade gracefully:\ntext-only requests work with just the TF-IDF baseline; fusion needs the meme text model,\nthe image model and (optionally) the detector plus models/fusion_lr.joblib.\n"""\n\nfrom __future__ import annotations\n\nimport base64\nimport io\nimport time\nfrom contextlib import asynccontextmanager\nfrom pathlib import Path\n\nimport numpy as np\nfrom fastapi import FastAPI, File, Form, HTTPException, UploadFile\n\nfrom src.config import LABELS, MODELS\nfrom src.serve.schemas import Health, ModerateJSON, ModerateResponse, Reason\n\nMAX_IMAGE_BYTES = 5 * 1024 * 1024\nTEXT_ONLY_REVIEW = 0.5  # text-only: Harmful argmax but P(Harmful) below this -> Neutral (review)\n\n\nclass Moderator:\n    """Holds every loaded component and turns inputs into a decision."""\n\n    def __init__(self, text=None, meme_text=None, image=None, detector=None, fusion: dict | None = None):\n        self.text, self.meme_text, self.image, self.detector, self.fusion = text, meme_text, image, detector, fusion\n\n    @classmethod\n    def from_disk(cls, models_dir: Path = MODELS) -> Moderator:\n        """Load whichever components exist under models_dir; the rest stay None."""\n        from src.text.infer_text import TransformerScorer, load_text_scorer\n\n        text = load_text_scorer("auto", models_dir)\n        meme_text = image = detector = fusion = None\n        if (models_dir / "meme_text_distilroberta" / "config.json").exists():\n            meme_text = TransformerScorer(models_dir / "meme_text_distilroberta")\n        if (models_dir / "effnet_b0_memes.pt").exists():\n            from src.image.infer_image import ImageScorer\n\n            image = ImageScorer(models_dir / "effnet_b0_memes.pt")\n        if (models_dir / "yolov8s_weapons.pt").exists():\n            from src.image.infer_yolo import WeaponDetector\n\n            detector = WeaponDetector(models_dir / "yolov8s_weapons.pt")\n        if (models_dir / "fusion_lr.joblib").exists():\n            import joblib\n\n            fusion = joblib.load(models_dir / "fusion_lr.joblib")\n        return cls(text, meme_text, image, detector, fusion)\n\n    def components(self) -> dict:\n        return {\n            "text": getattr(self.text, "name", None),\n            "meme_text": getattr(self.meme_text, "name", None),\n            "image": "efficientnet_b0" if self.image else None,\n            "detector": "yolov8s_weapons" if self.detector else None,\n            "fusion": ",".join(self.fusion["features"]) if self.fusion else None,\n        }\n\n    # ------------------------------------------------------------------ decision\n    def moderate(self, text: str | None, image) -> dict:\n        """Score the inputs and return every response field except latency_ms."""\n        if not text and image is None:\n            raise ValueError("send text, an image, or both")\n        scores: dict[str, float] = {}\n        reasons: list[Reason] = []\n        det = None\n        if text:\n            if self.text is None:\n                raise RuntimeError("no text model loaded")\n            probs = self.text.predict_proba([text])[0]\n            for i, lab in enumerate(LABELS):\n                scores[f"text_{lab.lower()}"] = round(float(probs[i]), 4)\n            scores["p_text_jigsaw"] = float(probs[2])  # P(Harmful), also a fusion feature\n            if self.meme_text is not None and image is not None:\n                # binary meme-caption model: the last column is P(hateful)\n                scores["p_text"] = float(self.meme_text.predict_proba([text])[0][-1])\n        if image is not None:\n            if self.image is not None:\n                scores["p_image"] = float(self.image.predict([image])[0])\n            if self.detector is not None:\n                det = self.detector.features([image])[0]\n                scores["weapon_max_conf"] = det["weapon_max_conf"]\n                scores["weapon_present"] = float(det["weapon_present"])\n\n        if text:\n            reasons.append(\n                Reason(\n                    signal="text_jigsaw",\n                    score=round(scores["p_text_jigsaw"], 4),\n                    detail="P(Harmful) from the Jigsaw text model",\n                )\n            )\n        if "p_text" in scores:\n            reasons.append(Reason(signal="text", score=round(scores["p_text"], 4), detail="meme-caption model"))\n        if "p_image" in scores:\n            reasons.append(Reason(signal="image", score=round(scores["p_image"], 4), detail="EfficientNet-B0"))\n        if det is not None and det["weapon_max_conf"] > 0:\n            reasons.append(\n                Reason(\n                    signal="weapon",\n                    score=round(det["weapon_max_conf"], 4),\n                    detail=f"{det[\'weapon_class\']} ({\'present\' if det[\'weapon_present\'] else \'weak\'})",\n                )\n            )\n\n        mode, label, p = self._decide(text, image, scores, det)\n        reasons.sort(key=lambda r: r.score, reverse=True)\n        if mode == "fusion":\n            reasons.insert(0, Reason(signal="fusion", score=round(p, 4), detail="late-fusion P(harmful)"))\n        scores["p_harmful"] = p\n        return {\n            "label": label,\n            "p_harmful": round(p, 4),\n            "scores": {k: round(v, 4) for k, v in scores.items()},\n            "reasons": reasons,\n            "mode": mode,\n        }\n\n    def _decide(self, text, image, scores, det) -> tuple[str, str, float]:\n        """Return (mode, label, p_harmful).\n\n        Fusion runs when text, an image and every fusion input are present; otherwise the\n        decision falls back to the text model, then to the image model alone.\n        """\n        if self.fusion is not None and text and image is not None:\n            feats = self.fusion["features"]\n            needed = {f for f in feats if f != "text_x_image"}  # text_x_image is derived from p_text and p_image\n            if needed <= set(scores):\n                from src.fusion.calibrate import apply_temperature_binary\n                from src.fusion.features import features_from_scores\n                from src.fusion.threshold import Thresholds, decide\n\n                x = features_from_scores(scores, feats)\n                p_raw = float(self.fusion["model"].predict_proba(x)[0, 1])\n                # thresholds act on the raw fusion score; the temperature-scaled value is reported alongside\n                label = LABELS[int(decide([p_raw], Thresholds(**self.fusion["thresholds"]))[0])]\n                scores["p_harmful_calibrated"] = float(\n                    apply_temperature_binary([p_raw], self.fusion.get("temperature", 1.0))[0]\n                )\n                return "fusion", label, p_raw\n        if text:\n            probs = [scores[f"text_{lab.lower()}"] for lab in LABELS]\n            label = LABELS[int(np.argmax(probs))]\n            if label == "Harmful" and probs[2] < TEXT_ONLY_REVIEW:\n                label = "Neutral"\n            return "text", label, float(probs[2])\n        # image only: fixed 0.5 cut on the image model\n        p = scores.get("p_image", 0.0)\n        label = "Harmful" if p >= 0.5 else "Safe"\n        if det is not None and det["weapon_present"] and label == "Safe":\n            label = "Neutral"  # weapon seen but image model unsure: send to review\n        return "image", label, float(p)\n\n\n# ---------------------------------------------------------------------- helpers\ndef _decode_image(raw: bytes):\n    from PIL import Image\n\n    if len(raw) > MAX_IMAGE_BYTES:\n        raise HTTPException(413, "image larger than 5 MB")\n    try:\n        return Image.open(io.BytesIO(raw)).convert("RGB")\n    except Exception as e:  # noqa: BLE001\n        raise HTTPException(400, f"could not decode image: {e}") from e\n\n\ndef _fetch(url: str) -> bytes:\n    """Download image_url (http or https only, 10 s timeout)."""\n    import httpx\n\n    if not url.startswith(("http://", "https://")):\n        raise HTTPException(400, "image_url must be http(s)")\n    try:\n        r = httpx.get(url, timeout=10, follow_redirects=True)\n        r.raise_for_status()\n    except httpx.HTTPError as e:\n        raise HTTPException(400, f"could not fetch image_url: {e}") from e\n    return r.content\n\n\ndef create_app(moderator: Moderator | None = None) -> FastAPI:\n    """Build the app. Tests inject a Moderator with stub models; otherwise models load from disk at startup."""\n    state: dict = {"mod": moderator}\n\n    def mod() -> Moderator:\n        if state["mod"] is None:\n            state["mod"] = Moderator.from_disk()\n        return state["mod"]\n\n    @asynccontextmanager\n    async def lifespan(_app: FastAPI):\n        mod()  # load every model once, before the first request\n        yield\n\n    app = FastAPI(title="Multimodal Content Moderation", version="1.0.0", lifespan=lifespan)\n\n    def run(text: str | None, image) -> dict:\n        t0 = time.perf_counter()\n        # bad input -> 422, missing model -> 503\n        try:\n            out = mod().moderate(text, image)\n        except ValueError as e:\n            raise HTTPException(422, str(e)) from e\n        except RuntimeError as e:\n            raise HTTPException(503, str(e)) from e\n        out["latency_ms"] = round((time.perf_counter() - t0) * 1000, 2)\n        return out\n\n    @app.get("/health", response_model=Health)\n    def health():\n        return {"status": "ok", "components": mod().components()}\n\n    @app.post("/moderate", response_model=ModerateResponse)\n    async def moderate(\n        text: str | None = Form(None), image_url: str | None = Form(None), image: UploadFile | None = File(None)\n    ):\n        img = None\n        if image is not None:\n            img = _decode_image(await image.read())\n        elif image_url:\n            img = _decode_image(_fetch(image_url))\n        return run(text, img)\n\n    @app.post("/moderate/json", response_model=ModerateResponse)\n    def moderate_json(body: ModerateJSON):\n        img = None\n        if body.image_b64:\n            img = _decode_image(base64.b64decode(body.image_b64))\n        elif body.image_url:\n            img = _decode_image(_fetch(body.image_url))\n        return run(body.text, img)\n\n    return app\n\n\napp = create_app()\n', 'src/serve/schemas.py': '"""Request/response contracts for the moderation API."""\n\nfrom __future__ import annotations\n\nfrom typing import Literal\n\nfrom pydantic import BaseModel, Field\n\n\nclass ModerateJSON(BaseModel):\n    text: str | None = Field(None, max_length=10_000)\n    image_url: str | None = None\n    image_b64: str | None = Field(None, description="base64-encoded image bytes")\n\n\nclass Reason(BaseModel):\n    signal: Literal["text", "text_jigsaw", "image", "weapon", "fusion"]\n    score: float\n    detail: str | None = None\n\n\nclass ModerateResponse(BaseModel):\n    label: Literal["Safe", "Neutral", "Harmful"]\n    p_harmful: float\n    scores: dict[str, float]\n    reasons: list[Reason]\n    mode: Literal["fusion", "text", "image"]\n    latency_ms: float\n\n\nclass Health(BaseModel):\n    status: Literal["ok"]\n    components: dict[str, str | None]\n', 'src/text/__init__.py': '', 'src/text/baselines.py': '"""Text baselines on Jigsaw: (a) VADER compound score + two thresholds, (b) TF-IDF + logistic regression.\n\nUsage:\n    python -m src.text.baselines                # both baselines, default "mild" mapping\n    python -m src.text.baselines --skip-ablation\n\nWrites results/text_eval.json (keys: vader, tfidf_logreg, ...), results/calibration.json\n(key: tfidf_logreg_text), per-item predictions in data/processed/ and models/tfidf_logreg.joblib.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport time\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nfrom src.config import DATA_PROCESSED, LABELS, MODELS, RESULTS, set_seed, write_json\nfrom src.eval.metrics import binary_report, classification_report, harmful_vs_rest\nfrom src.fusion.calibrate import ece_multiclass, fit_temperature_logits\nfrom src.text.clean import normalise\nfrom src.text.data import load_jigsaw\nfrom src.text.tfidf import TfidfLogReg\n\n\n# ------------------------------------------------------------------ VADER\ndef vader_scores(texts) -> np.ndarray:\n    """VADER compound score in [-1, 1] per text, on the first 2000 characters after normalisation."""\n    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer\n\n    sia = SentimentIntensityAnalyzer()\n    return np.array([sia.polarity_scores(normalise(t)[:2000])["compound"] for t in texts])\n\n\ndef vader_predict(compound: np.ndarray, t_harm: float, t_neutral: float) -> np.ndarray:\n    """compound < t_harm -> Harmful, t_harm <= compound < t_neutral -> Neutral, else Safe."""\n    y = np.zeros(len(compound), dtype=int)\n    y[compound < t_neutral] = 1\n    y[compound < t_harm] = 2\n    return y\n\n\ndef fit_vader(compound_val, y_val) -> tuple[float, float, float]:\n    """Grid-search the two thresholds (step 0.05, t_harm <= t_neutral) on validation for macro-F1.\n\n    Returns (best macro-F1, t_harm, t_neutral).\n    """\n    from sklearn.metrics import f1_score\n\n    grid = np.round(np.arange(-0.95, 0.96, 0.05), 2)\n    best = (-1.0, 0.0, 0.0)\n    for a in grid:\n        for b in grid[grid >= a]:\n            f = f1_score(y_val, vader_predict(compound_val, a, b), average="macro", labels=[0, 1, 2], zero_division=0)\n            if f > best[0]:\n                best = (float(f), float(a), float(b))\n    return best\n\n\ndef _save_preds(name: str, split: str, df: pd.DataFrame, probs: np.ndarray | None, pred: np.ndarray, extra=None):\n    """Write per-item predictions to data/processed/text_preds_<name>_<split>.csv."""\n    out = pd.DataFrame({"id": df["id"], "label": df["label"], "pred": pred})\n    if probs is not None:\n        for i, lab in enumerate(LABELS):\n            out[f"p_{lab.lower()}"] = probs[:, i]\n    if extra is not None:\n        for k, v in extra.items():\n            out[k] = v\n    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)\n    out.to_csv(DATA_PROCESSED / f"text_preds_{name}_{split}.csv", index=False)\n\n\ndef run_vader(data) -> dict:\n    t0 = time.time()\n    c_val, c_test = vader_scores(data["val"]["text"]), vader_scores(data["test"]["text"])\n    f_val, a, b = fit_vader(c_val, data["val"]["label"].to_numpy())\n    pred = vader_predict(c_test, a, b)\n    _save_preds("vader", "test", data["test"], None, pred, {"compound": c_test})\n    rep = classification_report(data["test"]["label"], pred)\n    # negated compound, so a higher score means more harmful\n    rep["harmful_vs_rest"] = binary_report(harmful_vs_rest(data["test"]["label"]), -c_test, threshold=-a)\n    return {\n        "thresholds": {"harmful_below": a, "neutral_below": b},\n        "val_macro_f1": round(f_val, 4),\n        "test": rep,\n        "seconds": round(time.time() - t0, 1),\n    }\n\n\ndef run_majority(data) -> dict:\n    """Always predict the most frequent training class (reference floor for macro-F1)."""\n    maj = int(np.bincount(data["train"]["label"]).argmax())\n    return {\n        "predicts": LABELS[maj],\n        "test": classification_report(data["test"]["label"], np.full(len(data["test"]), maj)),\n    }\n\n\ndef run_tfidf(data, save_model: bool = True) -> tuple[dict, dict]:\n    """Fit TF-IDF + LR on train. Returns (evaluation report, calibration report)."""\n    t0 = time.time()\n    model = TfidfLogReg().fit(data["train"]["text"].tolist(), data["train"]["label"].to_numpy())\n    fit_s = time.time() - t0\n    p_val = model.predict_proba(data["val"]["text"].tolist())\n    p_test = model.predict_proba(data["test"]["text"].tolist())\n    pred = p_test.argmax(1)\n    rep = classification_report(data["test"]["label"], pred)\n    rep["harmful_vs_rest"] = binary_report(harmful_vs_rest(data["test"]["label"]), p_test[:, 2])\n    out = {\n        "val": classification_report(data["val"]["label"], p_val.argmax(1)),\n        "test": rep,\n        "train_seconds": round(fit_s, 1),\n    }\n    if save_model:\n        MODELS.mkdir(parents=True, exist_ok=True)\n        joblib.dump(model, MODELS / "tfidf_logreg.joblib", compress=3)\n        _save_preds("tfidf", "val", data["val"], p_val, p_val.argmax(1))\n        _save_preds("tfidf", "test", data["test"], p_test, pred)\n    # calibration: temperature on log-probabilities (valid logits up to a constant)\n    lv, lt = np.log(np.clip(p_val, 1e-9, 1)), np.log(np.clip(p_test, 1e-9, 1))\n    t = fit_temperature_logits(data["val"]["label"].to_numpy(), lv)\n    from scipy.special import softmax\n\n    cal = {\n        "temperature": round(t, 4),\n        "val": {\n            "ece_before": round(ece_multiclass(data["val"]["label"], p_val), 4),\n            "ece_after": round(ece_multiclass(data["val"]["label"], softmax(lv / t, 1)), 4),\n        },\n        "test": {\n            "ece_before": round(ece_multiclass(data["test"]["label"], p_test), 4),\n            "ece_after": round(ece_multiclass(data["test"]["label"], softmax(lt / t, 1)), 4),\n        },\n        "note": "Top-label ECE, 15 bins, temperature fitted on validation.",\n    }\n    return out, cal\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--scheme", default="mild", choices=["mild", "spec"])\n    ap.add_argument("--skip-ablation", action="store_true")\n    args = ap.parse_args()\n    set_seed()\n\n    data = load_jigsaw(args.scheme)\n    sizes = {\n        k: {"n": len(v), "label_counts": {LABELS[i]: int((v["label"] == i).sum()) for i in range(3)}}\n        for k, v in data.items()\n    }\n    print("splits", sizes)\n\n    majority = run_majority(data)\n    vader = run_vader(data)\n    print("VADER test macro-F1", vader["test"]["macro_f1"])\n    tfidf, cal = run_tfidf(data)\n    print("TF-IDF test macro-F1", tfidf["test"]["macro_f1"])\n\n    payload = {\n        "dataset": "Jigsaw Toxic Comment Classification (Kaggle train.csv; official test.csv + test_labels.csv, rows with -1 dropped)",\n        "label_scheme": args.scheme,\n        "splits": sizes,\n        "majority_class": majority,\n        "vader": vader,\n        "tfidf_logreg": tfidf,\n    }\n    if not args.skip_ablation:\n        other = "spec" if args.scheme == "mild" else "mild"\n        d2 = load_jigsaw(other)\n        abl, _ = run_tfidf(d2, save_model=False)\n        payload[f"ablation_{other}_mapping"] = {\n            "splits": {k: {LABELS[i]: int((v["label"] == i).sum()) for i in range(3)} for k, v in d2.items()},\n            "tfidf_logreg_test": abl["test"],\n        }\n        print(f"TF-IDF ({other} mapping) test macro-F1", abl["test"]["macro_f1"])\n\n    write_json(RESULTS / "text_eval.json", payload)\n    write_json(RESULTS / "calibration.json", {"tfidf_logreg_text": cal})\n\n\nif __name__ == "__main__":\n    main()\n', 'src/text/clean.py': '"""Text normalisation shared by the baselines, the transformer and the API."""\n\nfrom __future__ import annotations\n\nimport html\nimport re\n\n_URL = re.compile(r"(https?://\\S+|www\\.\\S+)", re.IGNORECASE)\n_USER = re.compile(r"@\\w+")\n_IP = re.compile(r"\\b\\d{1,3}(?:\\.\\d{1,3}){3}\\b")\n_WIKI_META = re.compile(r"\\(talk\\)|\\(UTC\\)|\\bUTC\\b", re.IGNORECASE)\n_REPEAT = re.compile(r"(.)\\1{3,}")  # "soooooo" -> "sooo"\n_NON_TEXT = re.compile(r"[^a-z0-9\'!?.,\\s*]")\n_SPACE = re.compile(r"\\s+")\n\n\ndef normalise(text: str | None) -> str:\n    """Light cleaning that keeps casing and punctuation (used for the transformer and VADER).\n\n    VADER uses capitalisation and "!" as intensity cues, and subword models handle casing,\n    so we only strip markup, links, user handles, IPs and Wikipedia signature noise.\n    """\n    if text is None:\n        return ""\n    text = html.unescape(str(text))\n    text = _URL.sub(" ", text)\n    text = _USER.sub(" ", text)\n    text = _IP.sub(" ", text)\n    text = _WIKI_META.sub(" ", text)\n    text = _REPEAT.sub(r"\\1\\1\\1", text)\n    return _SPACE.sub(" ", text).strip()\n\n\ndef clean_for_tfidf(text: str | None) -> str:\n    """Aggressive cleaning for bag-of-words models: lowercase and drop symbols.\n\n    \'*\' is kept on purpose so masked profanity such as "f*ck" stays one token.\n    """\n    text = normalise(text).lower()\n    text = _NON_TEXT.sub(" ", text)\n    return _SPACE.sub(" ", text).strip()\n', 'src/text/data.py': '"""Load public text datasets and map them to Safe / Neutral / Harmful.\n\nLabel mapping for Jigsaw (see DECISIONS.md, D1):\n\n* ``mild`` (default)\n    Harmful = severe_toxic | threat | insult | identity_hate | (toxic & obscene)\n    Neutral = exactly one of {toxic, obscene} and nothing else ("mild" toxicity)\n    Safe    = no label set\n* ``spec`` (the literal first-draft mapping)\n    Harmful = toxic | severe_toxic | threat | insult | identity_hate\n    Neutral = obscene only\n    Safe    = no label set\n  Only ~0.2% of rows are Neutral under ``spec``, so it is kept as an ablation only.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\n\nfrom src.config import DATA_RAW, LABEL2ID, SEED\n\nJIGSAW_COLS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]\nJIGSAW_URL = (\n    "https://raw.githubusercontent.com/nicknochnack/CommentToxicity/main/"\n    "jigsaw-toxic-comment-classification-challenge/{name}.csv/{name}.csv"\n)\n\n\ndef map_jigsaw(df: pd.DataFrame, scheme: str = "mild") -> np.ndarray:\n    """Return integer labels (0 Safe, 1 Neutral, 2 Harmful) for Jigsaw rows."""\n    y = np.full(len(df), LABEL2ID["Safe"], dtype=int)\n    t, s, o, th, ins, ih = (df[c].to_numpy() == 1 for c in JIGSAW_COLS)\n    if scheme == "mild":\n        harmful = s | th | ins | ih | (t & o)\n        neutral = (t ^ o) & ~harmful  # exactly one of toxic, obscene\n    elif scheme == "spec":\n        harmful = t | s | th | ins | ih\n        neutral = o & ~harmful\n    else:\n        raise ValueError(f"unknown scheme {scheme!r}")\n    y[neutral] = LABEL2ID["Neutral"]\n    y[harmful] = LABEL2ID["Harmful"]\n    return y\n\n\ndef _read(name: str, raw_dir: Path) -> pd.DataFrame:\n    """Read a Jigsaw CSV, downloading it from JIGSAW_URL on first use."""\n    path = raw_dir / "jigsaw" / f"{name}.csv"\n    if not path.exists():\n        path.parent.mkdir(parents=True, exist_ok=True)\n        pd.read_csv(JIGSAW_URL.format(name=name)).to_csv(path, index=False)\n    return pd.read_csv(path)\n\n\ndef load_jigsaw(\n    scheme: str = "mild", val_size: float = 0.1, raw_dir: Path = DATA_RAW, seed: int = SEED\n) -> dict[str, pd.DataFrame]:\n    """Kaggle train.csv -> stratified train/val; official test.csv + test_labels.csv -> test.\n\n    Test rows labelled -1 (not scored in the competition) are dropped.\n    Each frame has columns: id, text, label.\n    """\n    train = _read("train", raw_dir)\n    test = _read("test", raw_dir).merge(_read("test_labels", raw_dir), on="id")\n    test = test[test["toxic"] != -1].reset_index(drop=True)\n\n    frames = {}\n    for name, df in (("train_full", train), ("test", test)):\n        frames[name] = pd.DataFrame(\n            {"id": df["id"], "text": df["comment_text"].astype(str), "label": map_jigsaw(df, scheme)}\n        )\n    tr, va = train_test_split(\n        frames["train_full"], test_size=val_size, random_state=seed, stratify=frames["train_full"]["label"]\n    )\n    return {\n        "train": tr.reset_index(drop=True),\n        "val": va.reset_index(drop=True),\n        "test": frames["test"],\n    }\n\n\n# ---------------------------------------------------------------- out-of-domain sets\nYOUTOXIC_HARM = [\n    "IsAbusive",\n    "IsThreat",\n    "IsHatespeech",\n    "IsRacist",\n    "IsNationalist",\n    "IsSexist",\n    "IsHomophobic",\n    "IsReligiousHate",\n    "IsRadicalism",\n]\n\n\ndef map_youtoxic(df: pd.DataFrame) -> np.ndarray:\n    """YouToxic English 1000 labels.\n\n    Toxic with any YOUTOXIC_HARM flag -> Harmful, toxic without one (e.g. only obscene or\n    provocative) -> Neutral, not toxic -> Safe.\n    """\n\n    def flag(col: str) -> np.ndarray:\n        return df[col].astype(str).str.lower().isin(["true", "1"]).to_numpy() if col in df else np.zeros(len(df), bool)\n\n    toxic = flag("IsToxic")\n    harm = np.zeros(len(df), bool)\n    for c in YOUTOXIC_HARM:\n        harm |= flag(c)\n    y = np.full(len(df), LABEL2ID["Safe"], dtype=int)\n    y[toxic] = LABEL2ID["Neutral"]\n    y[toxic & harm] = LABEL2ID["Harmful"]\n    return y\n\n\ndef load_youtoxic(path: Path) -> pd.DataFrame:\n    df = pd.read_csv(path)\n    return pd.DataFrame(\n        {"id": df.get("CommentId", df.index), "text": df["Text"].astype(str), "label": map_youtoxic(df)}\n    )\n\n\ndef map_civil(df: pd.DataFrame) -> np.ndarray:\n    """Civil Comments labels from fractional annotator scores.\n\n    toxicity >= 0.5 is toxic (Neutral); toxic and any of severe_toxicity, threat, insult or\n    identity_attack >= 0.5 -> Harmful.\n    """\n    tox = df["toxicity"].to_numpy() >= 0.5\n    harm = np.zeros(len(df), bool)\n    for c in ("severe_toxicity", "threat", "insult", "identity_attack"):\n        harm |= df[c].to_numpy() >= 0.5\n    y = np.full(len(df), LABEL2ID["Safe"], dtype=int)\n    y[tox] = LABEL2ID["Neutral"]\n    y[tox & harm] = LABEL2ID["Harmful"]\n    return y\n', 'src/text/eval_ood.py': '"""Out-of-domain text evaluation: models trained on Jigsaw, tested on YouTube comments.\n\nUsage:\n    python -m src.text.eval_ood --youtoxic path/to/youtoxic_english_1000.csv\n    python -m src.text.eval_ood --civil 5000          # fallback: Civil Comments test slice (HF datasets)\n\nEvery text model found in models/ (TF-IDF, transformers) plus VADER (with the thresholds\nlearned on Jigsaw validation) is scored. Writes results/text_ood.json.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.config import MODELS, RESULTS, SEED, read_json, write_json\nfrom src.eval.metrics import binary_report, classification_report, toxic_vs_safe\nfrom src.text.data import load_youtoxic, map_civil\nfrom src.text.infer_text import TfidfScorer, TransformerScorer\n\n\ndef load_civil(n: int) -> pd.DataFrame:\n    """Seeded random slice of n rows from the Civil Comments test split."""\n    from datasets import load_dataset\n\n    ds = load_dataset("google/civil_comments", split="test").shuffle(seed=SEED).select(range(n))\n    df = ds.to_pandas()\n    return pd.DataFrame({"id": np.arange(len(df)), "text": df["text"].astype(str), "label": map_civil(df)})\n\n\ndef evaluate(df: pd.DataFrame) -> dict:\n    """3-class report per model, plus toxic-vs-safe AUROC with 1 - P(Safe) as the score."""\n    out: dict = {}\n    y = df["label"].to_numpy()\n    texts = df["text"].tolist()\n    scorers = []\n    if (MODELS / "tfidf_logreg.joblib").exists():\n        scorers.append(TfidfScorer(MODELS / "tfidf_logreg.joblib"))\n    for d in sorted(MODELS.glob("text_*")):\n        if (d / "config.json").exists():\n            scorers.append(TransformerScorer(d))\n    for s in scorers:\n        p = s.predict_proba(texts)\n        rep = classification_report(y, p.argmax(1))\n        rep["toxic_vs_safe"] = binary_report(toxic_vs_safe(y), 1 - p[:, 0])\n        out[s.name] = rep\n    te = RESULTS / "text_eval.json"\n    if te.exists() and "vader" in read_json(te):\n        from src.text.baselines import vader_predict, vader_scores\n\n        th = read_json(te)["vader"]["thresholds"]\n        c = vader_scores(texts)\n        rep = classification_report(y, vader_predict(c, th["harmful_below"], th["neutral_below"]))\n        # negated compound, cut at the Neutral threshold learned on Jigsaw\n        rep["toxic_vs_safe"] = binary_report(toxic_vs_safe(y), -c, threshold=-th["neutral_below"])\n        out["vader"] = rep\n    return out\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--youtoxic", type=Path)\n    ap.add_argument("--civil", type=int, default=0)\n    args = ap.parse_args()\n    if args.youtoxic:\n        df, name = load_youtoxic(args.youtoxic), "YouToxic English 1000"\n    elif args.civil:\n        df, name = load_civil(args.civil), f"Civil Comments test slice (n={args.civil}, seed {SEED})"\n    else:\n        raise SystemExit("pass --youtoxic PATH or --civil N")\n    counts = {lab: int((df["label"] == i).sum()) for i, lab in enumerate(["Safe", "Neutral", "Harmful"])}\n    res = evaluate(df)\n    key = "youtoxic" if args.youtoxic else "civil_comments"\n    write_json(\n        RESULTS / "text_ood.json",\n        {\n            key: {\n                "dataset": name,\n                "label_counts": counts,\n                "models": res,\n                "note": "Models trained on Jigsaw only; no tuning on this set.",\n            }\n        },\n    )\n    for k, v in res.items():\n        print(f"{k:>16}: macro-F1 {v[\'macro_f1\']}  toxic-vs-safe AUROC {v[\'toxic_vs_safe\'].get(\'auroc\')}")\n\n\nif __name__ == "__main__":\n    main()\n', 'src/text/export_onnx.py': '"""Optional: export the text model to ONNX, apply dynamic INT8 quantization, and compare on CPU.\n\nUsage:\n    python -m src.text.export_onnx [--model-dir models/text_distilroberta] [--n-eval 5000] [--n-latency 300]\n\nWrites models/onnx/text_fp32.onnx, models/onnx/text_int8.onnx and results/onnx.json with\nper-item CPU latency (batch 1) for PyTorch FP32 vs ONNX INT8 and macro-F1 of both on the same\nstratified Jigsaw test sample.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom src.config import MODELS, RESULTS, SEED, set_seed, write_json\nfrom src.eval.metrics import classification_report\nfrom src.text.data import load_jigsaw\nfrom src.text.infer_text import OnnxScorer, TransformerScorer\n\n\ndef export(model_dir: Path, out: Path) -> tuple[Path, Path]:\n    """Export to ONNX (opset 17, dynamic batch and sequence axes), then quantize weights to INT8.\n\n    Returns (fp32 path, int8 path).\n    """\n    import torch\n    from onnxruntime.quantization import QuantType, quantize_dynamic\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n    out.mkdir(parents=True, exist_ok=True)\n    tok = AutoTokenizer.from_pretrained(model_dir)\n    model = AutoModelForSequenceClassification.from_pretrained(model_dir).float().eval()\n    enc = tok(["an example sentence"], return_tensors="pt")\n    fp32, int8 = out / "text_fp32.onnx", out / "text_int8.onnx"\n    kwargs = {}\n    if "dynamo" in torch.onnx.export.__code__.co_varnames:\n        # classic TorchScript exporter: needs no onnxscript and is stable for BERT-style models\n        kwargs["dynamo"] = False\n    torch.onnx.export(\n        model,\n        (enc["input_ids"], enc["attention_mask"]),\n        str(fp32),\n        input_names=["input_ids", "attention_mask"],\n        output_names=["logits"],\n        dynamic_axes={"input_ids": {0: "b", 1: "s"}, "attention_mask": {0: "b", 1: "s"}, "logits": {0: "b"}},\n        opset_version=17,\n        **kwargs,\n    )\n    quantize_dynamic(str(fp32), str(int8), weight_type=QuantType.QInt8)\n    return fp32, int8\n\n\ndef per_item_ms(scorer, texts) -> list[float]:\n    """Batch-1 latency per text in ms, after 10 warm-up calls."""\n    for t in texts[:10]:\n        scorer.predict_proba([t])\n    out = []\n    for t in texts:\n        t0 = time.perf_counter()\n        scorer.predict_proba([t])\n        out.append((time.perf_counter() - t0) * 1000)\n    return out\n\n\ndef main() -> None:\n    import torch\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model-dir", type=Path, default=MODELS / "text_distilroberta")\n    ap.add_argument("--n-eval", type=int, default=5000)\n    ap.add_argument("--n-latency", type=int, default=300)\n    args = ap.parse_args()\n    set_seed()\n    fp32, int8 = export(args.model_dir, MODELS / "onnx")\n\n    test = load_jigsaw()["test"]\n    from sklearn.model_selection import train_test_split\n\n    sample, _ = train_test_split(test, train_size=args.n_eval, random_state=SEED, stratify=test["label"])\n    texts, y = sample["text"].tolist(), sample["label"].to_numpy()\n    lat_texts = test["text"].sample(args.n_latency, random_state=SEED + 1).tolist()\n\n    pt = TransformerScorer(args.model_dir, device="cpu")\n    ox = OnnxScorer(int8, args.model_dir)\n    f_pt = classification_report(y, pt.predict_proba(texts, batch=64).argmax(1))["macro_f1"]\n    f_ox = classification_report(y, ox.predict_proba(texts, batch=64).argmax(1))["macro_f1"]\n    l_pt, l_ox = per_item_ms(pt, lat_texts), per_item_ms(ox, lat_texts)\n    res = {\n        "model_dir": str(args.model_dir),\n        "eval_sample": {"n": len(y), "source": "stratified Jigsaw test sample"},\n        "pytorch_fp32": {\n            "macro_f1": f_pt,\n            "p50_ms": round(float(np.percentile(l_pt, 50)), 2),\n            "p95_ms": round(float(np.percentile(l_pt, 95)), 2),\n        },\n        "onnx_int8": {\n            "macro_f1": f_ox,\n            "p50_ms": round(float(np.percentile(l_ox, 50)), 2),\n            "p95_ms": round(float(np.percentile(l_ox, 95)), 2),\n        },\n        "speedup_p50": round(float(np.percentile(l_pt, 50) / np.percentile(l_ox, 50)), 2),\n        "macro_f1_change": round(f_ox - f_pt, 4),\n        "size_mb": {"fp32": round(fp32.stat().st_size / 1e6, 1), "int8": round(int8.stat().st_size / 1e6, 1)},\n        "torch_threads": torch.get_num_threads(),\n        "batch_size": 1,\n    }\n    write_json(RESULTS / "onnx.json", res, merge=False)\n    print(res)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/text/infer_text.py': '"""Text scorer used by the API and the OOD evaluation.\n\nLoads the fine-tuned transformer from models/text_<name>/ if present (optionally an\nONNX INT8 export), otherwise falls back to the TF-IDF + LR baseline in\nmodels/tfidf_logreg.joblib. Every scorer returns probabilities over LABELS.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom src.config import MODELS\nfrom src.text.clean import normalise\n\n\nclass TransformerScorer:\n    """Fine-tuned Hugging Face classifier (PyTorch, CPU or GPU)."""\n\n    backend = "transformer"\n\n    def __init__(self, model_dir: Path, max_len: int = 128, device: str | None = None):\n        import torch\n        from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n        self.torch = torch\n        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        self.tok = AutoTokenizer.from_pretrained(model_dir)\n        # .float() upcasts checkpoints saved in fp16 (scripts/save_half.py)\n        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir).float().to(self.device).eval()\n        self.max_len = max_len\n        self.name = Path(model_dir).name\n\n    def predict_proba(self, texts: list[str], batch: int = 32) -> np.ndarray:\n        out = []\n        with self.torch.inference_mode():\n            for i in range(0, len(texts), batch):\n                enc = self.tok(\n                    [normalise(t) for t in texts[i : i + batch]],\n                    truncation=True,\n                    max_length=self.max_len,\n                    padding=True,\n                    return_tensors="pt",\n                ).to(self.device)\n                out.append(self.torch.softmax(self.model(**enc).logits, -1).cpu().numpy())\n        return np.concatenate(out) if out else np.zeros((0, 3))\n\n\nclass OnnxScorer:\n    """ONNX Runtime classifier on CPU; tok_dir holds the matching tokenizer."""\n\n    backend = "onnx"\n\n    def __init__(self, onnx_path: Path, tok_dir: Path, max_len: int = 128):\n        import onnxruntime as ort\n        from transformers import AutoTokenizer\n\n        self.sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])\n        self.tok = AutoTokenizer.from_pretrained(tok_dir)\n        self.max_len = max_len\n        self.name = Path(onnx_path).stem\n        # tokenizer outputs not declared by the exported graph are dropped before each run\n        self.inputs = {i.name for i in self.sess.get_inputs()}\n\n    def predict_proba(self, texts: list[str], batch: int = 32) -> np.ndarray:\n        from scipy.special import softmax\n\n        out = []\n        for i in range(0, len(texts), batch):\n            enc = self.tok(\n                [normalise(t) for t in texts[i : i + batch]],\n                truncation=True,\n                max_length=self.max_len,\n                padding=True,\n                return_tensors="np",\n            )\n            feed = {k: v.astype(np.int64) for k, v in enc.items() if k in self.inputs}\n            out.append(softmax(self.sess.run(None, feed)[0], -1))\n        return np.concatenate(out) if out else np.zeros((0, 3))\n\n\nclass TfidfScorer:\n    backend = "tfidf"\n    name = "tfidf_logreg"\n\n    def __init__(self, path: Path):\n        import joblib\n\n        self.model = joblib.load(path)\n\n    def predict_proba(self, texts: list[str], batch: int = 0) -> np.ndarray:\n        return self.model.predict_proba(list(texts))\n\n\ndef load_text_scorer(prefer: str = "auto", models_dir: Path = MODELS):\n    """Return a text scorer, or None if no model is found.\n\n    prefer: auto | onnx | transformer | tfidf. With auto the order is ONNX INT8, then the\n    transformer (DistilRoBERTa if several exist), then TF-IDF.\n    """\n    onnx = models_dir / "onnx" / "text_int8.onnx"\n    tdirs = sorted(p for p in models_dir.glob("text_*") if (p / "config.json").exists())\n    if prefer in ("auto", "onnx") and onnx.exists() and tdirs:\n        return OnnxScorer(onnx, tdirs[0])\n    if prefer in ("auto", "transformer") and tdirs:\n        pick = next((p for p in tdirs if "distilroberta" in p.name), tdirs[0])\n        return TransformerScorer(pick)\n    if (models_dir / "tfidf_logreg.joblib").exists():\n        return TfidfScorer(models_dir / "tfidf_logreg.joblib")\n    return None\n', 'src/text/tfidf.py': '"""TF-IDF + logistic regression text classifier (kept in its own module so pickles load anywhere)."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom scipy.sparse import hstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\n\nfrom src.config import SEED\nfrom src.text.clean import clean_for_tfidf\n\n\nclass TfidfLogReg:\n    """Word (1-2 gram) + char_wb (2-5 gram) TF-IDF features into class-weighted multinomial LR."""\n\n    def __init__(self, max_word: int = 200_000, max_char: int = 200_000, C: float = 4.0):\n        self.word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=max_word, sublinear_tf=True)\n        self.char = TfidfVectorizer(\n            analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=max_char, sublinear_tf=True\n        )\n        self.clf = LogisticRegression(C=C, class_weight="balanced", max_iter=2000, random_state=SEED)\n\n    def _x(self, texts, fit: bool = False):\n        clean = [clean_for_tfidf(t) for t in texts]\n        if fit:\n            return hstack([self.word.fit_transform(clean), self.char.fit_transform(clean)]).tocsr()\n        return hstack([self.word.transform(clean), self.char.transform(clean)]).tocsr()\n\n    def fit(self, texts, y):\n        self.clf.fit(self._x(texts, fit=True), y)\n        return self\n\n    def predict_proba(self, texts) -> np.ndarray:\n        return self.clf.predict_proba(self._x(texts))\n', 'src/text/train_meme_text.py': '"""Text-only Hateful Memes branch: DistilRoBERTa fine-tuned on meme captions (binary). GPU.\n\nAlso scores meme captions with the Jigsaw text model (if models/text_distilroberta exists) as an\nextra, zero-shot feature p_text_jigsaw.\n\nUsage:\n    python -m src.text.train_meme_text [--memes-root PATH] [--epochs 4]\nWrites models/meme_text_distilroberta/ and data/processed/meme_scores_text.csv.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport math\n\nimport pandas as pd\nfrom sklearn.model_selection import train_test_split\n\nfrom src.config import DATA_PROCESSED, MODELS, RESULTS, SEED, set_seed, write_json\nfrom src.image.memes import load_memes\n\n\ndef main() -> None:\n    import torch\n    from datasets import Dataset\n    from scipy.special import softmax\n    from sklearn.metrics import roc_auc_score\n    from transformers import (\n        AutoModelForSequenceClassification,\n        AutoTokenizer,\n        DataCollatorWithPadding,\n        Trainer,\n        TrainingArguments,\n    )\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--memes-root")\n    ap.add_argument("--model", default="distilroberta-base")\n    ap.add_argument("--epochs", type=float, default=4)\n    args = ap.parse_args()\n    set_seed()\n    splits, meta = load_memes(args.memes_root)\n    tok = AutoTokenizer.from_pretrained(args.model)\n    # 10% of branch_train is held out to pick the best epoch by AUROC\n    tr, es = train_test_split(\n        splits["branch_train"], test_size=0.1, random_state=SEED, stratify=splits["branch_train"]["label"]\n    )\n\n    def ds(df):\n        d = Dataset.from_dict({"text": df["text"].astype(str).tolist(), "label": df["label"].tolist()})\n        return d.map(lambda b: tok(b["text"], truncation=True, max_length=64), batched=True, remove_columns=["text"])\n\n    model = AutoModelForSequenceClassification.from_pretrained(args.model, num_labels=2)\n    out_dir = MODELS / "meme_text_distilroberta"\n    targs = TrainingArguments(\n        output_dir=str(out_dir / "ckpt"),\n        num_train_epochs=args.epochs,\n        learning_rate=2e-5,\n        per_device_train_batch_size=32,\n        per_device_eval_batch_size=128,\n        warmup_steps=int(math.ceil(len(tr) / 32) * args.epochs * 0.1),\n        weight_decay=0.01,\n        fp16=torch.cuda.is_available(),\n        seed=SEED,\n        eval_strategy="epoch",\n        save_strategy="epoch",\n        load_best_model_at_end=True,\n        metric_for_best_model="auroc",\n        save_total_limit=1,\n        report_to=[],\n        logging_steps=50,\n    )\n\n    def metrics(p):\n        return {"auroc": roc_auc_score(p.label_ids, softmax(p.predictions, 1)[:, 1])}\n\n    trainer = Trainer(\n        model=model,\n        args=targs,\n        train_dataset=ds(tr),\n        eval_dataset=ds(es),\n        data_collator=DataCollatorWithPadding(tok),\n        compute_metrics=metrics,\n    )\n    trainer.train()\n    trainer.save_model(str(out_dir))\n    tok.save_pretrained(str(out_dir))\n\n    jig = None\n    jdir = MODELS / "text_distilroberta"\n    if (jdir / "config.json").exists():\n        from src.text.infer_text import TransformerScorer\n\n        jig = TransformerScorer(jdir)\n\n    rows = []\n    for split in ("fusion_train", "test"):\n        df = splits[split]\n        p = softmax(trainer.predict(ds(df)).predictions, 1)[:, 1]\n        row = pd.DataFrame({"id": df["id"], "split": split, "label": df["label"], "p_text": p})\n        if jig is not None:\n            row["p_text_jigsaw"] = jig.predict_proba(df["text"].astype(str).tolist())[:, 2]\n        rows.append(row)\n    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)\n    pd.concat(rows).to_csv(DATA_PROCESSED / "meme_scores_text.csv", index=False)\n    write_json(\n        RESULTS / "training_log.json",\n        {\n            "meme_text": {\n                "log": [h for h in trainer.state.log_history if "eval_auroc" in h],\n                "jigsaw_feature": jig is not None,\n                "data": meta,\n            }\n        },\n    )\n    write_json(RESULTS / "data_summary.json", {"hateful_memes": meta})\n\n\nif __name__ == "__main__":\n    main()\n', 'src/text/train_text.py': '"""Fine-tune DistilRoBERTa (or RoBERTa) on Jigsaw with a class-weighted loss. GPU recommended.\n\nUsage (Kaggle / Colab):\n    python -m src.text.train_text --model distilroberta-base --epochs 2\n    python -m src.text.train_text --model roberta-base --epochs 2      # optional size comparison\n\nWrites models/text_<short>/, per-item predictions in data/processed/, and merges\n<short> into results/text_eval.json and <short>_text into results/calibration.json.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport math\nimport time\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.config import DATA_PROCESSED, LABELS, MODELS, RESULTS, SEED, set_seed, write_json\nfrom src.eval.metrics import binary_report, classification_report, harmful_vs_rest\nfrom src.fusion.calibrate import ece_multiclass, fit_temperature_logits\nfrom src.text.clean import normalise\nfrom src.text.data import load_jigsaw\n\n\ndef short_name(model: str) -> str:\n    """e.g. "distilroberta-base" -> "distilroberta"."""\n    return model.split("/")[-1].replace("-base", "").replace("-", "_")\n\n\ndef class_weights(y: np.ndarray, power: float = 0.5) -> np.ndarray:\n    """Inverse-frequency weights softened by `power` (0.5 = sqrt) and normalised to mean 1."""\n    counts = np.bincount(y, minlength=len(LABELS)).astype(float)\n    w = (counts.sum() / np.maximum(counts, 1)) ** power\n    return w / w.mean()\n\n\ndef main() -> None:\n    import torch\n    from datasets import Dataset\n    from transformers import (\n        AutoModelForSequenceClassification,\n        AutoTokenizer,\n        DataCollatorWithPadding,\n        Trainer,\n        TrainingArguments,\n    )\n\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", default="distilroberta-base")\n    ap.add_argument("--epochs", type=float, default=2)\n    ap.add_argument("--max-len", type=int, default=128)\n    ap.add_argument("--batch", type=int, default=64)\n    ap.add_argument("--lr", type=float, default=3e-5)\n    ap.add_argument("--scheme", default="mild")\n    ap.add_argument("--limit", type=int, default=0, help="subsample train for smoke tests")\n    args = ap.parse_args()\n    set_seed()\n    name = short_name(args.model)\n\n    data = load_jigsaw(args.scheme)\n    if args.limit:\n        data["train"] = data["train"].sample(args.limit, random_state=SEED)\n    tok = AutoTokenizer.from_pretrained(args.model)\n\n    def to_ds(df: pd.DataFrame) -> Dataset:\n        ds = Dataset.from_dict({"text": [normalise(t) for t in df["text"]], "label": df["label"].tolist()})\n        return ds.map(\n            lambda b: tok(b["text"], truncation=True, max_length=args.max_len), batched=True, remove_columns=["text"]\n        )\n\n    ds = {k: to_ds(v) for k, v in data.items()}\n    weights = torch.tensor(class_weights(data["train"]["label"].to_numpy()), dtype=torch.float)\n\n    # Trainer with class-weighted cross-entropy in place of the default loss\n    class WeightedTrainer(Trainer):\n        def compute_loss(self, model, inputs, return_outputs=False, **kw):\n            labels = inputs.pop("labels")\n            outputs = model(**inputs)\n            loss = torch.nn.functional.cross_entropy(outputs.logits, labels, weight=weights.to(outputs.logits.device))\n            return (loss, outputs) if return_outputs else loss\n\n    model = AutoModelForSequenceClassification.from_pretrained(\n        args.model,\n        num_labels=len(LABELS),\n        id2label=dict(enumerate(LABELS)),\n        label2id={lab: i for i, lab in enumerate(LABELS)},\n    )\n    out_dir = MODELS / f"text_{name}"\n    targs = TrainingArguments(\n        output_dir=str(out_dir / "ckpt"),\n        num_train_epochs=args.epochs,\n        learning_rate=args.lr,\n        per_device_train_batch_size=args.batch,\n        per_device_eval_batch_size=args.batch * 2,\n        warmup_steps=int(math.ceil(len(ds["train"]) / args.batch) * args.epochs * 0.06),  # 6% of all steps\n        weight_decay=0.01,\n        fp16=torch.cuda.is_available(),\n        seed=SEED,\n        eval_strategy="epoch",\n        save_strategy="no",\n        logging_steps=200,\n        report_to=[],\n    )\n    trainer = WeightedTrainer(\n        model=model,\n        args=targs,\n        train_dataset=ds["train"],\n        eval_dataset=ds["val"],\n        data_collator=DataCollatorWithPadding(tok),\n    )\n    t0 = time.time()\n    trainer.train()\n    train_s = time.time() - t0\n\n    logits = {k: trainer.predict(ds[k]).predictions for k in ("val", "test")}\n    from scipy.special import softmax\n\n    probs = {k: softmax(v, 1) for k, v in logits.items()}\n    for split in ("val", "test"):\n        df = pd.DataFrame({"id": data[split]["id"], "label": data[split]["label"], "pred": probs[split].argmax(1)})\n        for i, lab in enumerate(LABELS):\n            df[f"p_{lab.lower()}"] = probs[split][:, i]\n            df[f"logit_{lab.lower()}"] = logits[split][:, i]\n        DATA_PROCESSED.mkdir(parents=True, exist_ok=True)\n        df.to_csv(DATA_PROCESSED / f"text_preds_{name}_{split}.csv", index=False)\n\n    rep = classification_report(data["test"]["label"], probs["test"].argmax(1))\n    rep["harmful_vs_rest"] = binary_report(harmful_vs_rest(data["test"]["label"]), probs["test"][:, 2])\n    write_json(\n        RESULTS / "text_eval.json",\n        {\n            name: {\n                "base_model": args.model,\n                "epochs": args.epochs,\n                "max_len": args.max_len,\n                "lr": args.lr,\n                "batch": args.batch,\n                "class_weights": weights.tolist(),\n                "train_rows": len(data["train"]),\n                "val": classification_report(data["val"]["label"], probs["val"].argmax(1)),\n                "test": rep,\n                "train_seconds": round(train_s, 1),\n                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",\n            }\n        },\n    )\n\n    t = fit_temperature_logits(data["val"]["label"].to_numpy(), logits["val"])\n    write_json(\n        RESULTS / "calibration.json",\n        {\n            f"{name}_text": {\n                "temperature": round(t, 4),\n                "val": {\n                    "ece_before": round(ece_multiclass(data["val"]["label"], probs["val"]), 4),\n                    "ece_after": round(ece_multiclass(data["val"]["label"], softmax(logits["val"] / t, 1)), 4),\n                },\n                "test": {\n                    "ece_before": round(ece_multiclass(data["test"]["label"], probs["test"]), 4),\n                    "ece_after": round(ece_multiclass(data["test"]["label"], softmax(logits["test"] / t, 1)), 4),\n                },\n                "note": "Top-label ECE, 15 bins, temperature fitted on validation logits.",\n            }\n        },\n    )\n\n    trainer.save_model(str(out_dir))\n    tok.save_pretrained(str(out_dir))\n    (out_dir / "temperature.txt").write_text(f"{t:.6f}\\n")\n    print(f"{name}: test macro-F1 {rep[\'macro_f1\']}  saved to {out_dir}")\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/__init__.py': '', 'scripts/add_jigsaw_feature.py': '"""Add the zero-shot Jigsaw-model score (p_text_jigsaw) to the meme caption scores on CPU.\n\nUsed when the text and image notebooks run in parallel, so the image notebook had no Jigsaw model.\nNeeds models/text_distilroberta/ and the meme split files in data/raw/hateful_memes/.\n\nUsage:\n    python -m scripts.add_jigsaw_feature\n"""\n\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom src.config import DATA_PROCESSED, DATA_RAW, MODELS\nfrom src.image.memes import load_memes\nfrom src.text.infer_text import TransformerScorer\n\n\ndef main() -> None:\n    path = DATA_PROCESSED / "meme_scores_text.csv"\n    df = pd.read_csv(path)\n    if "p_text_jigsaw" in df.columns:\n        print("p_text_jigsaw already present; nothing to do")\n        return\n    splits, _ = load_memes(DATA_RAW / "hateful_memes")\n    captions = pd.concat([splits["fusion_train"], splits["test"]])[["id", "text"]].drop_duplicates("id")\n    captions["id"] = captions["id"].astype(int)  # jsonl ids may be zero-padded strings; score CSVs store ints\n    df["id"] = df["id"].astype(int)\n    df = df.merge(captions, on="id", how="left")\n    if df["text"].isna().any():\n        raise SystemExit("some meme ids have no caption; check data/raw/hateful_memes")\n    scorer = TransformerScorer(MODELS / "text_distilroberta", device="cpu")\n    df["p_text_jigsaw"] = scorer.predict_proba(df["text"].astype(str).tolist(), batch=64)[:, 2]\n    df.drop(columns="text").to_csv(path, index=False)\n    print(f"added p_text_jigsaw to {len(df)} rows")\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/check_dashes.py': '"""Fail if any tracked text file contains an em dash or en dash (house style)."""\n\nfrom __future__ import annotations\n\nimport sys\n\nfrom src.config import ROOT\n\nSKIP_DIRS = {".git", ".venv", "venv", "data", "models", "__pycache__", ".ruff_cache", ".pytest_cache"}\nEXTS = {".py", ".md", ".ipynb", ".json", ".yml", ".yaml", ".toml", ".txt", ".cfg", ""}\nBAD = (chr(0x2014), chr(0x2013))  # em dash, en dash\n\n\ndef main() -> None:\n    hits = []\n    for p in ROOT.rglob("*"):\n        if p.is_dir() or SKIP_DIRS & set(p.relative_to(ROOT).parts) or p.suffix not in EXTS:\n            continue\n        try:\n            text = p.read_text(encoding="utf-8")\n        except (UnicodeDecodeError, OSError):\n            continue\n        for i, line in enumerate(text.splitlines(), 1):\n            if any(b in line for b in BAD):\n                hits.append(f"{p.relative_to(ROOT)}:{i}")\n    if hits:\n        print("dashes found:\\n" + "\\n".join(hits))\n        sys.exit(1)\n    print("OK: no em or en dashes")\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/data_summary.py': '"""Record dataset versions, licences, sizes and splits in results/data_summary.json (Jigsaw part).\n\nThe GPU notebooks add the hateful_memes and open_images_weapons entries; eval_ood adds the OOD set.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\n\nfrom src.config import DATA_RAW, LABELS, RESULTS, SEED, write_json\nfrom src.text.data import JIGSAW_URL, load_jigsaw\n\n\ndef sha256(path) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef main() -> None:\n    data = load_jigsaw("mild")\n    files = {n: sha256(DATA_RAW / "jigsaw" / f"{n}.csv") for n in ("train", "test", "test_labels")}\n    write_json(\n        RESULTS / "data_summary.json",\n        {\n            "jigsaw": {\n                "name": "Jigsaw Toxic Comment Classification Challenge (Kaggle, 2018)",\n                "licence": "Comment text CC0; underlying Wikipedia talk-page text CC BY-SA 3.0",\n                "download": JIGSAW_URL.format(name="<train|test|test_labels>")\n                + " (byte-identical mirror of the Kaggle files; see sha256)",\n                "sha256": files,\n                "label_mapping": "mild (see src/text/data.py and DECISIONS.md D1)",\n                "split_protocol": f"train.csv -> 90/10 stratified train/val (seed {SEED}); test.csv + test_labels.csv (rows != -1) -> test",\n                "splits": {\n                    k: {"n": len(v), **{LABELS[i]: int((v["label"] == i).sum()) for i in range(3)}}\n                    for k, v in data.items()\n                },\n            }\n        },\n    )\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/merge_results.py': '"""Unpack results_*.zip files from the GPU notebooks into the repo.\n\nUsage:\n    python -m scripts.merge_results results_text.zip results_image.zip\n\nJSON files under results/ are merged key by key (so CPU-side keys such as the Jigsaw\nablation are kept); every other file (weights, score CSVs, plots) is written in place.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport sys\nimport zipfile\nfrom pathlib import Path\n\nfrom src.config import ROOT\n\n\ndef merge(zip_path: Path) -> list[str]:\n    done = []\n    with zipfile.ZipFile(zip_path) as z:\n        for info in z.infolist():\n            if info.is_dir():\n                continue\n            dest = ROOT / info.filename\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            raw = z.read(info)\n            if dest.suffix == ".json" and dest.parent.name == "results" and dest.exists():\n                current = json.loads(dest.read_text(encoding="utf-8"))\n                current.update(json.loads(raw))\n                dest.write_text(json.dumps(current, indent=2) + "\\n", encoding="utf-8")\n            else:\n                dest.write_bytes(raw)\n            done.append(info.filename)\n    return done\n\n\ndef main() -> None:\n    if len(sys.argv) < 2:\n        raise SystemExit(__doc__)\n    for p in sys.argv[1:]:\n        files = merge(Path(p))\n        print(f"{p}: {len(files)} files")\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/render_readme.py': '"""Fill the README result tables from results/*.json (the only allowed source of numbers).\n\nUsage:\n    python -m scripts.render_readme\n\nReplaces everything between <!-- RESULTS:<name> --> and <!-- /RESULTS:<name> --> markers.\nA block whose result file does not exist yet shows a "pending" line instead of numbers.\n"""\n\nfrom __future__ import annotations\n\nimport re\n\nfrom src.config import RESULTS, ROOT, read_json\n\nNICE = {\n    "majority_class": "Majority class (always Safe)",\n    "vader": "VADER compound + thresholds",\n    "tfidf_logreg": "TF-IDF + logistic regression",\n    "distilroberta": "DistilRoBERTa (fine-tuned)",\n    "roberta": "RoBERTa-base (fine-tuned)",\n    "text_only": "Text only (DistilRoBERTa on captions)",\n    "image_only": "Image only (EfficientNet-B0)",\n    "image_plus_weapon": "Image + YOLOv8s weapon features",\n    "late_fusion": "Late fusion: text + image + weapon (served by the API)",\n    "clip_linear_probe": "CLIP ViT-L/14 linear probe (image + caption)",\n    "text_plus_clip": "Text + CLIP fusion",\n    "late_fusion_plus_clip": "Late fusion + CLIP",\n}\n\n\ndef f(x, pct=False):\n    if x is None:\n        return "n/a"\n    return f"{100 * x:.1f}" if pct else f"{x:.3f}"\n\n\ndef load(name):\n    p = RESULTS / f"{name}.json"\n    return read_json(p) if p.exists() else None\n\n\ndef pending(what):\n    return f"_Pending: {what}._"\n\n\ndef block_text():\n    d = load("text_eval")\n    if not d:\n        return pending("run `make baselines`")\n    rows = [\n        "| Model | Macro-F1 | Harmful F1 | Neutral F1 | Safe F1 | Harmful AUROC | Accuracy |",\n        "|---|---|---|---|---|---|---|",\n    ]\n    for k in ("majority_class", "vader", "tfidf_logreg", "distilroberta", "roberta"):\n        if k in d:\n            t = d[k]["test"]\n            pc = t["per_class"]\n            auc = t.get("harmful_vs_rest", {}).get("auroc")\n            rows.append(\n                f"| {NICE[k]} | {f(t[\'macro_f1\'])} | {f(pc[\'Harmful\'][\'f1\'])} | {f(pc[\'Neutral\'][\'f1\'])} | "\n                f"{f(pc[\'Safe\'][\'f1\'])} | {f(auc)} | {f(t[\'accuracy\'])} |"\n            )\n    n = d["splits"]["test"]["n"]\n    rows.append(f"\\nJigsaw official test set, n = {n:,}. Source: `results/text_eval.json`.")\n    if "distilroberta" not in d:\n        rows.append(pending("DistilRoBERTa row appears after the text notebook"))\n    return "\\n".join(rows)\n\n\ndef block_ood():\n    d = load("text_ood")\n    if not d:\n        return pending("out-of-domain evaluation (text notebook, step 4)")\n    rows = []\n    for key in ("youtoxic", "civil_comments"):\n        if key not in d:\n            continue\n        ds = d[key]\n        rows += [f"**{ds[\'dataset\']}**", "", "| Model | Macro-F1 | Toxic vs safe AUROC |", "|---|---|---|"]\n        for k, v in ds["models"].items():\n            name = NICE.get(k.replace("text_", ""), k)\n            rows.append(f"| {name} | {f(v[\'macro_f1\'])} | {f(v[\'toxic_vs_safe\'].get(\'auroc\'))} |")\n        rows.append("")\n    rows.append("Models trained on Jigsaw only, no tuning on these sets. Source: `results/text_ood.json`.")\n    return "\\n".join(rows)\n\n\ndef block_fusion():\n    d = load("fusion_eval")\n    if not d:\n        return pending("image notebook (Hateful Memes)")\n    head = d.get("headline")\n    rows = ["| Model | AUROC | Avg. precision | Macro-F1 |", "|---|---|---|---|"]\n    for k, v in d["variants"].items():\n        t = v["test"]\n        name = NICE.get(k, k)\n        if k == head:\n            name = f"**{name}** (selected on validation)"\n        rows.append(f"| {name} | {f(t.get(\'auroc\'))} | {f(t.get(\'average_precision\'))} | {f(t[\'macro_f1\'])} |")\n    ds = load("data_summary") or {}\n    hm = ds.get("hateful_memes", {})\n    split = hm.get("test_file", "see data_summary.json")\n    n = hm.get("sizes", {}).get("test", {}).get("n")\n    rows.append(\n        f"\\nHateful Memes `{split}` (n = {n}), never used for training or selection. The headline model is the one with "\n        "the best out-of-fold AUROC on the held-out fusion split. Source: `results/fusion_eval.json`."\n    )\n    return "\\n".join(rows)\n\n\ndef block_detector():\n    d = load("detector_eval")\n    if not d:\n        return pending("YOLOv8s training (image notebook, step 3)")\n    t = d["test"]\n    per = ", ".join(f"{k} {v:.3f}" for k, v in t["per_class_mAP50_95"].items())\n    return (\n        "| mAP50 | mAP50-95 | Precision | Recall |\\n|---|---|---|---|\\n"\n        f"| {f(t[\'mAP50\'])} | {f(t[\'mAP50_95\'])} | {f(t[\'precision\'])} | {f(t[\'recall\'])} |\\n\\n"\n        f"Per-class mAP50-95: {per}. Source: `results/detector_eval.json`."\n    )\n\n\ndef block_triage():\n    th, tr = load("threshold"), load("triage")\n    if not (th and tr):\n        return pending("fusion step")\n    rows = [\n        "| Model | Harmful recall (val) | Harmful recall (test) | Harmful precision (test) | Auto-cleared (test) |",\n        "|---|---|---|---|---|",\n    ]\n    head = th.get("variant")\n    for k, v in th.get("per_variant", {}).items():\n        name = NICE.get(k, k)\n        cleared = f"{v[\'test_auto_cleared_pct\']:.1f}%"\n        if k == head:\n            name, cleared = f"**{name}**", f"**{cleared}**"\n        rows.append(\n            f"| {name} | {f(v[\'oof_harmful_recall\'], True)}% | {f(v[\'test_harmful_recall\'], True)}% | "\n            f"{f(v[\'test_harmful_precision\'], True)}% | {cleared} |"\n        )\n    ref = tr.get("reference_test_oracle_at_target_recall", {}).get("auto_cleared_pct")\n    rows.append(\n        f"\\nFor the headline model, {tr[\'auto_cleared_pct\']:.1f}% of test items fall below t_clear = "\n        f"{th[\'thresholds\'][\'t_clear\']:.3f} and would skip review; {tr[\'harmful_rate_in_cleared_pct\']}% of that cleared pile "\n        f"is actually hateful (base rate {tr[\'base_harmful_rate_pct\']}%)."\n        + (\n            f" Setting the threshold on the test labels themselves to force exactly 90% recall would clear {ref:.1f}% "\n            "(shown only to size the validation-to-test gap)."\n            if ref is not None\n            else ""\n        )\n        + " Sources: `results/threshold.json`, `results/triage.json`."\n    )\n    return "\\n".join(rows)\n\n\ndef block_calibration():\n    d = load("calibration")\n    if not d:\n        return pending("calibration")\n    rows = ["| Component | Temperature | Test ECE before | Test ECE after |", "|---|---|---|---|"]\n    labels = {\n        "tfidf_logreg_text": "TF-IDF text model (3-class)",\n        "distilroberta_text": "DistilRoBERTa (3-class)",\n        "roberta_text": "RoBERTa-base (3-class)",\n    }\n    for k, v in d.items():\n        name = labels.get(k) or ("Fusion: " + NICE.get(k.replace("fusion_", ""), k) if k.startswith("fusion_") else k)\n        rows.append(f"| {name} | {v[\'temperature\']:.2f} | {f(v[\'test\'][\'ece_before\'])} | {f(v[\'test\'][\'ece_after\'])} |")\n    rows.append("\\nSource: `results/calibration.json`.")\n    return "\\n".join(rows)\n\n\ndef block_latency():\n    d, o = load("latency"), load("onnx")\n    if not d and not o:\n        return pending("latency benchmark on the laptop (`make serve`, then `make latency`)")\n    rows = []\n    if d:\n        rows += ["| Request type | p50 (ms) | p95 (ms) | Throughput (req/s) | Hardware |", "|---|---|---|---|---|"]\n        mode_names = {\n            "text": "Text only (ONNX INT8 DistilRoBERTa)",\n            "fusion": "Text + image (all branches + late fusion)",\n        }\n        for k, v in d.items():\n            hw = v["hardware"].get("cpu") or v["hardware"].get("machine")\n            rows.append(\n                f"| {mode_names.get(k, k)} | {v[\'client_ms\'][\'p50\']:.1f} | {v[\'client_ms\'][\'p95\']:.1f} | "\n                f"{v[\'throughput_rps\'][\'value\']:.1f} | {hw}, {v[\'hardware\'][\'logical_cores\']} vCPU |"\n            )\n    if o:\n        rows += [\n            "",\n            "| Text model on CPU (batch 1) | p50 (ms) | Macro-F1 |",\n            "|---|---|---|",\n            f"| PyTorch FP32 | {o[\'pytorch_fp32\'][\'p50_ms\']:.1f} | {f(o[\'pytorch_fp32\'][\'macro_f1\'])} |",\n            f"| ONNX Runtime INT8 | {o[\'onnx_int8\'][\'p50_ms\']:.1f} | {f(o[\'onnx_int8\'][\'macro_f1\'])} |",\n            f"\\nINT8 speed-up {o[\'speedup_p50\']:.2f}x, macro-F1 change {o[\'macro_f1_change\']:+.4f}. Source: `results/onnx.json`.",\n        ]\n    return "\\n".join(rows)\n\n\nBLOCKS = {\n    "text": block_text,\n    "ood": block_ood,\n    "fusion": block_fusion,\n    "detector": block_detector,\n    "triage": block_triage,\n    "calibration": block_calibration,\n    "latency": block_latency,\n}\n\n\ndef main() -> None:\n    path = ROOT / "README.md"\n    s = path.read_text(encoding="utf-8")\n    for name, fn in BLOCKS.items():\n        pat = re.compile(rf"(<!-- RESULTS:{name} -->).*?(<!-- /RESULTS:{name} -->)", re.S)\n        s = pat.sub(lambda m, fn=fn: m.group(1) + "\\n" + fn() + "\\n" + m.group(2), s)\n    path.write_text(s, encoding="utf-8")\n    print("README tables refreshed from results/")\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/results_summary.py': '"""Write results/RESULTS_SUMMARY.md (every headline number and its source file) plus a local bullet summary.\n\nUsage:\n    python -m scripts.results_summary      # also prints the table\n\nNumbers come only from results/*.json. Anything not produced yet is shown as PENDING.\n"""\n\nfrom __future__ import annotations\n\nfrom src.config import RESULTS, read_json\n\nP = "PENDING"\n\n\ndef load(name):\n    p = RESULTS / f"{name}.json"\n    return read_json(p) if p.exists() else {}\n\n\ndef get(d, *keys):\n    for k in keys:\n        if not isinstance(d, dict) or k not in d:\n            return None\n        d = d[k]\n    return d\n\n\ndef fmt(x, spec=".3f", suffix=""):\n    return P if x is None else f"{x:{spec}}{suffix}"\n\n\ndef main() -> None:\n    te, ds, fe = load("text_eval"), load("data_summary"), load("fusion_eval")\n    th, tr, lat, ox, de, ood = (\n        load(n) for n in ("threshold", "triage", "latency", "onnx", "detector_eval", "text_ood")\n    )\n\n    n_train = get(ds, "jigsaw", "splits", "train", "n")\n    n_val = get(ds, "jigsaw", "splits", "val", "n")\n    n_jig = (n_train + n_val) if n_train and n_val else None\n    n_fit = get(te, "distilroberta", "train_rows") or n_train  # rows the model was actually fitted on\n    f_dr = get(te, "distilroberta", "test", "macro_f1")\n    f_tf = get(te, "tfidf_logreg", "test", "macro_f1")\n    f_va = get(te, "vader", "test", "macro_f1")\n    h_dr = get(te, "distilroberta", "test", "per_class", "Harmful", "f1")\n    h_tf = get(te, "tfidf_logreg", "test", "per_class", "Harmful", "f1")\n    h_va = get(te, "vader", "test", "per_class", "Harmful", "f1")\n    a_dr = get(te, "distilroberta", "test", "harmful_vs_rest", "auroc")\n    a_tf = get(te, "tfidf_logreg", "test", "harmful_vs_rest", "auroc")\n\n    def pts(x, y):\n        return round(100 * (x - y), 1) if x is not None and y is not None else None\n\n    head = get(fe, "headline")\n    auc_t = get(fe, "variants", "text_only", "test", "auroc")\n    auc_i = get(fe, "variants", "image_only", "test", "auroc")\n    auc_lf = get(fe, "variants", "late_fusion", "test", "auroc")\n    auc_h = get(fe, "variants", head, "test", "auroc") if head else None\n    rec = get(th, "at_t_clear", "harmful_recall")\n    prec = get(th, "at_t_clear", "harmful_precision")\n    rec_val = get(th, "val_oof", "at_t_clear", "harmful_recall")\n    cleared = get(tr, "auto_cleared_pct")\n    t50, t95 = get(lat, "text", "client_ms", "p50"), get(lat, "text", "client_ms", "p95")\n    m50, m95 = get(lat, "fusion", "client_ms", "p50"), get(lat, "fusion", "client_ms", "p95")\n    cores = get(lat, "text", "hardware", "logical_cores")\n    spd, df1 = get(ox, "speedup_p50"), get(ox, "macro_f1_change")\n    map50, map5095 = get(de, "test", "mAP50"), get(de, "test", "mAP50_95")\n    n_memes = get(ds, "hateful_memes", "sizes", "test", "n")\n    ood_key = "youtoxic" if "youtoxic" in ood else "civil_comments"\n    ood_auc = get(ood, ood_key, "models", "text_distilroberta", "toxic_vs_safe", "auroc")\n    ood_auc_tf = get(ood, ood_key, "models", "tfidf_logreg", "toxic_vs_safe", "auroc")\n\n    def pct(x):\n        return None if x is None else 100 * x\n\n    rows = [\n        ("Jigsaw labelled comments (train + val)", fmt(n_jig, ","), "data_summary.json"),\n        ("Jigsaw comments the model was fitted on (train split)", fmt(n_fit, ","), "data_summary.json"),\n        ("Jigsaw test comments", fmt(get(ds, "jigsaw", "splits", "test", "n"), ","), "data_summary.json"),\n        (\n            "DistilRoBERTa macro-F1 / Harmful F1 / Harmful AUROC",\n            f"{fmt(f_dr)} / {fmt(h_dr)} / {fmt(a_dr)}",\n            "text_eval.json",\n        ),\n        (\n            "TF-IDF + LR macro-F1 / Harmful F1 / Harmful AUROC",\n            f"{fmt(f_tf)} / {fmt(h_tf)} / {fmt(a_tf)}",\n            "text_eval.json",\n        ),\n        ("VADER macro-F1 / Harmful F1", f"{fmt(f_va)} / {fmt(h_va)}", "text_eval.json"),\n        (\n            "Harmful F1 gain over TF-IDF / VADER (points)",\n            f"{fmt(pts(h_dr, h_tf), \'.1f\')} / {fmt(pts(h_dr, h_va), \'.1f\')}",\n            "text_eval.json",\n        ),\n        ("Macro-F1 gain over TF-IDF (points)", fmt(pts(f_dr, f_tf), ".1f"), "text_eval.json"),\n        (\n            f"Toxic-vs-safe AUROC out of domain ({ood_key}): DistilRoBERTa / TF-IDF",\n            f"{fmt(ood_auc)} / {fmt(ood_auc_tf)}",\n            "text_ood.json",\n        ),\n        ("Hateful Memes test size", fmt(n_memes, ","), "data_summary.json"),\n        (\n            "Hateful Memes AUROC: text only / image only / late fusion",\n            f"{fmt(auc_t)} / {fmt(auc_i)} / {fmt(auc_lf)}",\n            "fusion_eval.json",\n        ),\n        (f"Hateful Memes AUROC: headline model ({head})", fmt(auc_h), "fusion_eval.json"),\n        (\n            "Harmful recall at t_clear: validation (OOF) / test",\n            f"{fmt(pct(rec_val), \'.1f\', \'%\')} / {fmt(pct(rec), \'.1f\', \'%\')}",\n            "threshold.json",\n        ),\n        ("Harmful precision at t_clear (test)", fmt(pct(prec), ".1f", "%"), "threshold.json"),\n        ("Items auto-cleared, skipping manual review (test)", fmt(cleared, ".1f", "%"), "triage.json"),\n        ("API latency, text only: p50 / p95 (ms)", f"{fmt(t50, \'.1f\')} / {fmt(t95, \'.1f\')}", "latency.json"),\n        ("API latency, text + image: p50 / p95 (ms)", f"{fmt(m50, \'.1f\')} / {fmt(m95, \'.1f\')}", "latency.json"),\n        ("Benchmark CPU threads", fmt(cores, "d"), "latency.json"),\n        (\n            "ONNX INT8 speed-up vs PyTorch FP32 (p50) / macro-F1 change",\n            f"{fmt(spd, \'.2f\', \'x\')} / {fmt(df1, \'+.4f\')}",\n            "onnx.json",\n        ),\n        ("YOLOv8s weapons mAP50 / mAP50-95 (test)", f"{fmt(map50)} / {fmt(map5095)}", "detector_eval.json"),\n    ]\n\n    def b(x):\n        return f"**{x}**"\n\n    bullets = [\n        f"- Fine-tuned **DistilRoBERTa** on {b(f\'{round(n_fit / 1000)}K\' if n_fit else P)} comments to {b(fmt(h_dr, \'.2f\'))} Harmful F1, "\n        f"beating TF-IDF and VADER baselines by {b(fmt(pts(h_dr, h_tf), \'.0f\'))} and {b(fmt(pts(h_dr, h_va), \'.0f\'))} points.",\n        f"- Fused caption and CLIP ViT-L/14 signals, lifting Hateful Memes AUROC from {b(fmt(auc_t, \'.2f\'))} (text-only) "\n        f"to {b(fmt(auc_h, \'.2f\'))} on {fmt(n_memes, \',\')} held-out memes.",\n        f"- Tuned the Harmful threshold on validation for {b(\'90%\')} recall, auto-clearing {b(fmt(cleared, \'.0f\', \'%\'))} of test items "\n        f"from manual review ({fmt(pct(rec), \'.0f\', \'%\')} test recall).",\n        f"- Served via **FastAPI** at {b(fmt(t95, \'.0f\', \' ms\'))} p95 per text item on {fmt(cores, \'d\')} vCPUs; INT8 ONNX cut text latency "\n        f"{b(fmt(spd, \'.1f\', \'x\'))} with no F1 loss ({fmt(df1, \'+.3f\')}).",\n    ]\n\n    md = [\n        "# Results summary",\n        "",\n        "Every number below is read from the listed file in `results/` by `scripts/results_summary.py`.",\n        "",\n        "| Metric | Value | Source |",\n        "|---|---|---|",\n        *[f"| {m} | {v} | `results/{s}` |" for m, v, s in rows],\n    ]\n    (RESULTS / "RESULTS_SUMMARY.md").write_text("\\n".join(md).rstrip() + "\\n", encoding="utf-8")\n    # The bullet file is git-ignored and stays out of the public repo.\n    bl = ["# Resume bullets (local only, filled from RESULTS_SUMMARY.md)", "", *bullets]\n    (RESULTS / "RESUME_BULLETS.md").write_text("\\n".join(bl) + "\\n", encoding="utf-8")\n    print("\\n".join(md + ["", *bl]))\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/run_all.py': '"""Reproduce every CPU-side result file in order. GPU stages live in notebooks/.\n\nUsage:\n    python -m scripts.run_all                 # data summary, baselines, OOD (if data), fusion (if scores), plots\n    python -m scripts.run_all --latency       # also start the API and run the latency benchmark\n    python -m scripts.run_all --onnx          # also run the ONNX INT8 comparison (needs the text model)\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom src.config import DATA_PROCESSED, DATA_RAW, MODELS\n\n\ndef sh(*args: str) -> None:\n    print("\\n$", " ".join(args), flush=True)\n    subprocess.run([sys.executable, "-m", *args], check=True)\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--latency", action="store_true")\n    ap.add_argument("--onnx", action="store_true")\n    ap.add_argument("--skip-baselines", action="store_true")\n    args = ap.parse_args()\n\n    sh("scripts.data_summary")\n    if not args.skip_baselines:\n        sh("src.text.baselines")\n    yt = sorted((DATA_RAW / "youtoxic").glob("*.csv"))\n    if yt:\n        sh("src.text.eval_ood", "--youtoxic", str(yt[0]))\n    else:\n        print("\\n[skip] OOD eval: put the YouToxic csv in data/raw/youtoxic/ (or run eval_ood --civil N)")\n    if list(DATA_PROCESSED.glob("meme_scores_*.csv")):\n        sh("src.fusion.train_fusion")\n    else:\n        print("\\n[skip] fusion: no data/processed/meme_scores_*.csv yet (run the image notebook)")\n    if args.onnx and (MODELS / "text_distilroberta" / "config.json").exists():\n        sh("src.text.export_onnx")\n    if args.latency:\n        srv = subprocess.Popen([sys.executable, "-m", "uvicorn", "src.serve.app:app", "--port", "8000"])\n        try:\n            time.sleep(15)  # give uvicorn time to load the models\n            sh("src.eval.latency", "--mode", "text")\n            if (MODELS / "fusion_lr.joblib").exists() and Path(DATA_RAW / "hateful_memes").exists():\n                sh(\n                    "src.eval.latency",\n                    "--mode",\n                    "fusion",\n                    "--n",\n                    "300",\n                    "--memes-root",\n                    str(DATA_RAW / "hateful_memes"),\n                )\n        finally:\n            srv.terminate()\n    sh("src.eval.plots")\n    sh("scripts.render_readme")\n    sh("scripts.results_summary")\n\n\nif __name__ == "__main__":\n    main()\n', 'scripts/save_half.py': '"""Save an fp16 copy of a fine-tuned text classifier (halves the download; loading upcasts to fp32).\n\nUsage:\n    python -m scripts.save_half models/text_distilroberta pack/models/text_distilroberta\n\nRuns in its own process so it is unaffected by packages installed later in a notebook session.\n"""\n\nfrom __future__ import annotations\n\nimport shutil\nimport sys\nfrom pathlib import Path\n\n\ndef main() -> None:\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n    src, dst = Path(sys.argv[1]), Path(sys.argv[2])\n    model = AutoModelForSequenceClassification.from_pretrained(src).half()\n    model.save_pretrained(dst)\n    AutoTokenizer.from_pretrained(src).save_pretrained(dst)\n    for extra in src.glob("*.txt"):\n        shutil.copy(extra, dst)\n    print(f"saved fp16 copy of {src} to {dst}")\n\n\nif __name__ == "__main__":\n    main()\n'}
for rel, src in FILES.items():
    p = WORK / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(src, encoding="utf-8")
os.chdir(WORK); sys.path.insert(0, str(WORK))
os.environ["MM_ROOT"] = str(WORK)
print(f"wrote {len(FILES)} files to {WORK}")

In [ ]:
# Shell helper: prints each step with elapsed time and stops the run if a required step fails
from IPython import get_ipython
import time as _t
_T0 = _t.time()
def sh(cmd, required=True):
    ip = get_ipython()
    cmd = ip.var_expand(cmd, depth=1)
    print(f"\n$ {cmd}   [t+{(_t.time() - _T0) / 60:.1f} min]", flush=True)
    ip.system(cmd)
    code = ip.user_ns.get("_exit_code", 0)
    if code != 0:
        msg = f"FAILED ({code}): {cmd}"
        print(msg, flush=True)
        if required:
            raise RuntimeError(msg)

In [ ]:
sh('pip -q install timm kagglehub')
import glob, shutil, pathlib, torch
print(torch.__version__, torch.cuda.is_available())
# reuse the Jigsaw text model if the text notebook output is attached
for cfg in glob.glob('/kaggle/input/**/text_distilroberta/config.json', recursive=True):
    shutil.copytree(pathlib.Path(cfg).parent, 'models/text_distilroberta', dirs_exist_ok=True); print('found', cfg); break
for z in glob.glob('/kaggle/input/**/results_text.zip', recursive=True):
    sh('python -m scripts.merge_results {z}', required=False)
    break
# Hateful Memes (Phase 2 release, public Kaggle copy); licence accepted by the author
import os, kagglehub
if not glob.glob('/kaggle/input/**/train.jsonl', recursive=True):
    os.environ['MM_MEMES_ROOT'] = kagglehub.dataset_download('williamberrios/hateful-memes')
from src.image.memes import load_memes
splits, meta = load_memes(); print(meta)

## 1. Text-only branch on meme captions (about 5 minutes)

In [ ]:
sh('python -m src.text.train_meme_text --epochs 4')

## 2. Image-only branch: EfficientNet-B0 (about 15 minutes)

In [ ]:
sh('python -m src.image.train_effnet --epochs 8')

## 3. Optional stronger comparison: CLIP ViT-L/14 linear probe (about 5 minutes)

In [ ]:
RUN_CLIP = True
if RUN_CLIP:
    sh('python -m src.fusion.clip_probe', required=False)

## 4. Weapon detector: YOLOv8s on Open Images V7 (Handgun, Rifle, Shotgun, Knife)
Downloads about 4K images with FiftyOne, trains, and evaluates on the held-out test split (about 45 to 75 minutes). Set `FAST = True` for a quicker, smaller run.

In [ ]:
sh('pip -q install ultralytics fiftyone  # installed last so it cannot disturb earlier steps', required=False)
FAST = False
if FAST:
    sh('python -m src.image.train_yolo --per-class 400 --epochs 20', required=False)
else:
    sh('python -m src.image.train_yolo --per-class 1000 --epochs 40', required=False)
sh('cat results/detector_eval.json', required=False)

In [ ]:
sh('python -m src.image.infer_yolo', required=False)

## 5. Late fusion, threshold, triage, calibration (seconds, CPU)

In [ ]:
sh('python -m src.fusion.train_fusion')
sh('cat results/fusion_eval.json | head -c 2500')
sh('cat results/triage.json')

## 6. Package results and weights

In [ ]:
import glob, shutil, zipfile, pathlib
def save_half(src, dst):
    # fp16 copy in a separate process (immune to packages installed earlier in this session)
    sh(f'python -m scripts.save_half {src} {dst}')
out = pathlib.Path('/kaggle/working/results_image.zip') if pathlib.Path('/kaggle').exists() else pathlib.Path('results_image.zip')
keep = ['models/effnet_b0_memes.pt', 'models/yolov8s_weapons.pt', 'models/fusion_lr.joblib']
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in list(pathlib.Path('results').glob('*')) + list(pathlib.Path('data/processed').glob('meme_scores_*')):
        z.write(p)
    for k in keep:
        if pathlib.Path(k).exists(): z.write(k)
    save_half('models/meme_text_distilroberta', 'pack/models/meme_text_distilroberta')
    for p in pathlib.Path('pack').rglob('*'):
        if p.is_file(): z.write(p, p.relative_to('pack'))
    # local copies for the laptop (git-ignored): split files and test images for latency and examples
    from src.image.memes import load_memes
    s, meta = load_memes()
    root = pathlib.Path(meta['root'])
    for name in ['train.jsonl', meta['test_file']]:
        z.write(root / name, f'data/raw/hateful_memes/{name}')
    for p in s['test']['img_path']:
        z.write(p, 'data/raw/hateful_memes/img/' + pathlib.Path(p).name)
    # YouToxic English 1000 (attached input, CC BY 4.0) for the out-of-domain test on the laptop
    for p in glob.glob('/kaggle/input/**/youtoxic_english_1000.csv', recursive=True)[:1]:
        z.write(p, 'data/raw/youtoxic/youtoxic_english_1000.csv')
print(out, round(out.stat().st_size / 1e6, 1), 'MB')